# D2: State Machine Intake — Schema & Extraction Methods

## Цель эксперимента
Сравнить стратегии сбора структурированных данных (анамнеза) в стоматологическом чат-боте:
- **CS-A**: Free-form LLM (LLM сам решает что спрашивать)
- **CS-B**: Deterministic SM + Regex (фиксированный порядок, pattern matching)
- **CS-C**: Hybrid SM+LLM (SM управляет flow, LLM извлекает данные)

Дополнительно: сравнить **4 дизайна схемы** (SD-1..SD-4) и **методы извлечения** (EM-1..EM-6).

## Гипотезы
- **H₁**: Hybrid SM+LLM (C3) превосходит Free-form LLM (A3) по completion rate (p < 0.05)
- **H₂**: API Schema mode (EM-3) превосходит Regex (EM-1) по extraction accuracy
- **H₃**: Type-specific schema (SD-2) превосходит Universal (SD-1) по expert sufficiency
- **H₄**: Instructor (EM-4) не хуже API Schema (EM-3) по accuracy

## Критерии подтверждения
1. `completion_rate_C3 > completion_rate_A3` (p < 0.05, Wilcoxon)
2. `extraction_accuracy_EM3 > extraction_accuracy_EM1` (Bootstrap CI)
3. `expert_sufficient_rate >= 0.70` для лучшей комбинации
4. `avg_turns_diff <= 2` между стратегиями

## Метрики
| Метрика | Описание |
|---------|----------|
| Completion Rate | % заполненных required полей |
| Extraction Accuracy | Точность vs gold standard (exact + semantic) |
| Turns to Complete | Ходов до полного заполнения |
| Token Cost | Стоимость в токенах |
| Expert Sufficiency | % кейсов с completion ≥ 0.9 |
| Schema Compliance | % валидных JSON ответов |
| Robustness | Деградация при adversarial input |

## Фазы
- **Phase 1**: Setup + Data + Deterministic SM (B1) + Theorem + Monte Carlo (без API)
- **Phase 2**: LLM experiments (A3, A4, C3, C4 — требует API key)
- **Phase 3**: Schema comparison + Adversarial
- **Phase 4**: Statistics + Viz + Export

## 1. Setup

In [33]:
# === Imports ===
import sys
import os
import json
import re
import random
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass, field
from typing import Any, Dict, List, Literal, Optional, Tuple
from enum import Enum

import numpy as np
import pandas as pd
from tqdm import tqdm

# Визуализация
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

# Окружение
from dotenv import load_dotenv
load_dotenv()

# === Пути проекта ===
STUDY_ROOT = Path('.').resolve()
sys.path.insert(0, str(STUDY_ROOT))

# Утилиты study
from utils.schemas import (
    INTAKE_SCHEMA, COMPLAINT_TYPES, REQUIRED_BY_TYPE,
    get_required_fields, validate_intake, compute_completion_rate,
)
from utils.data import CASE_TEMPLATES, generate_d2_cases

# === Воспроизводимость ===
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# === Директории вывода ===
DATA_DIR = STUDY_ROOT / 'data'
OUTPUT_FIGURES = STUDY_ROOT / 'outputs' / 'figures'
OUTPUT_TABLES = STUDY_ROOT / 'outputs' / 'tables'
OUTPUT_REPORTS = STUDY_ROOT / 'outputs' / 'reports'
OUTPUT_DIAGRAMS = STUDY_ROOT / 'outputs' / 'diagrams'
CONFIGS_DIR = STUDY_ROOT / 'configs'

for p in [DATA_DIR, OUTPUT_FIGURES, OUTPUT_TABLES, OUTPUT_REPORTS, OUTPUT_DIAGRAMS, CONFIGS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# === Константы эксперимента ===
N_CASES = 10                    # Кейсов на эксперимент
MAX_TURNS_SM = 8                # Макс. ходов для SM
COOPERATION_RATE_DEFAULT = 0.9  # Вероятность кооперативного ответа пациента
N_MONTE_CARLO = 1000            # Итераций Monte Carlo
EXPERT_THRESHOLD = 0.9          # Порог expert sufficiency

print(f"Эксперимент D2 v3 | {datetime.now().isoformat()}")
print(f"Seed: {SEED} | Cases: {N_CASES} | Max turns: {MAX_TURNS_SM}")
print(f"Cooperation rate: {COOPERATION_RATE_DEFAULT} | Monte Carlo: {N_MONTE_CARLO}")

Эксперимент D2 v3 | 2026-03-19T20:58:11.825584
Seed: 42 | Cases: 10 | Max turns: 8
Cooperation rate: 0.9 | Monte Carlo: 1000


## 2. Дизайны схем (SD-1 .. SD-4)

Сравниваем 4 подхода к определению обязательных полей анамнеза:

| ID | Название | Полей | Описание |
|----|----------|-------|----------|
| SD-1 | Universal 4 | 4 | Production-aligned (symptoms, localization, duration, chronic_or_allergies) |
| SD-2 | Type-specific | 3-6 | Разные поля по типу жалобы (из `utils/schemas.py`) |
| SD-3 | Adaptive | 4→6 | Начинаем с SD-1, расширяем после определения типа |
| SD-4 | Flat Extended | 8 | Универсальные 8 полей для всех типов |

In [34]:
# === SD-1: Universal 4 (production-aligned) ===
# Источник: ai-core/apps/agent-llm/core/schemas.py:AnamnesisData
SD1_FIELDS = ["symptoms", "localization", "duration", "chronic_or_allergies"]

SD1_QUESTIONS: Dict[str, str] = {
    "symptoms": "Что вас беспокоит? Опишите симптомы.",
    "localization": "Где именно болит? Укажите место.",
    "duration": "Как давно это беспокоит?",
    "chronic_or_allergies": "Есть ли у вас хронические заболевания или аллергии?",
}

# === SD-2: Type-specific (из utils/schemas.py:REQUIRED_BY_TYPE) ===
SD2_REQUIRED_BY_TYPE: Dict[str, List[str]] = {
    "acute_pain":   ["chief_complaint", "localization", "duration", "intensity", "onset", "triggers"],
    "chronic_pain": ["chief_complaint", "localization", "duration", "intensity", "onset", "relievers"],
    "esthetics":    ["chief_complaint", "localization", "desired_outcome"],
    "ortho":        ["chief_complaint", "localization", "bite_issues"],
    "therapy":      ["chief_complaint", "localization", "last_visit"],
}

SD2_QUESTIONS: Dict[str, str] = {
    "chief_complaint": "Что вас беспокоит? Опишите вашу проблему.",
    "localization": "Где именно болит или беспокоит? Укажите зуб или область.",
    "duration": "Как давно это беспокоит?",
    "intensity": "Оцените интенсивность боли от 1 до 10.",
    "onset": "Когда это началось? Было ли что-то, что спровоцировало?",
    "triggers": "Что усиливает боль? (холодное, горячее, жевание)",
    "relievers": "Что облегчает состояние?",
    "desired_outcome": "Какой результат вы хотели бы получить?",
    "last_visit": "Когда вы последний раз были у стоматолога?",
    "bite_issues": "Есть ли проблемы с прикусом?",
}

# === SD-3: Adaptive (SD-1 → expand по типу) ===
# Маппинг: SD-1 поля → SD-2 расширения по типу жалобы
SD3_EXTENSIONS: Dict[str, List[str]] = {
    "acute_pain":   ["intensity", "onset", "triggers"],
    "chronic_pain": ["intensity", "onset", "relievers"],
    "esthetics":    ["desired_outcome"],
    "ortho":        ["bite_issues"],
    "therapy":      ["last_visit"],
}

def sd3_get_fields(complaint_type: Optional[str] = None) -> List[str]:
    """SD-3: базовые 4 поля + расширение при известном типе жалобы."""
    base = ["symptoms", "localization", "duration", "chronic_or_allergies"]
    if complaint_type and complaint_type in SD3_EXTENSIONS:
        return base + SD3_EXTENSIONS[complaint_type]
    return base

# === SD-4: Flat Extended (8 универсальных) ===
SD4_FIELDS = [
    "symptoms", "localization", "duration", "chronic_or_allergies",
    "onset", "triggers", "intensity", "medications",
]

SD4_QUESTIONS: Dict[str, str] = {
    **SD1_QUESTIONS,
    "onset": "Когда это началось?",
    "triggers": "Что усиливает боль или дискомфорт?",
    "intensity": "Оцените интенсивность боли от 1 до 10.",
    "medications": "Принимаете ли вы сейчас какие-либо лекарства?",
}

# === Сводная таблица ===
SCHEMA_DESIGNS = {
    "SD-1": {"name": "Universal 4", "fields_fn": lambda ct: SD1_FIELDS, "questions": SD1_QUESTIONS},
    "SD-2": {"name": "Type-specific", "fields_fn": lambda ct: SD2_REQUIRED_BY_TYPE.get(ct, ["chief_complaint", "localization"]), "questions": SD2_QUESTIONS},
    "SD-3": {"name": "Adaptive", "fields_fn": sd3_get_fields, "questions": {**SD1_QUESTIONS, **SD2_QUESTIONS, **SD4_QUESTIONS}},
    "SD-4": {"name": "Flat Extended 8", "fields_fn": lambda ct: SD4_FIELDS, "questions": SD4_QUESTIONS},
}

# Проверка
print("=== Дизайны схем ===")
for sd_id, sd in SCHEMA_DESIGNS.items():
    for ct in COMPLAINT_TYPES:
        fields = sd["fields_fn"](ct)
        print(f"  {sd_id} ({sd['name']}) | {ct}: {len(fields)} полей → {fields}")

=== Дизайны схем ===
  SD-1 (Universal 4) | acute_pain: 4 полей → ['symptoms', 'localization', 'duration', 'chronic_or_allergies']
  SD-1 (Universal 4) | chronic_pain: 4 полей → ['symptoms', 'localization', 'duration', 'chronic_or_allergies']
  SD-1 (Universal 4) | esthetics: 4 полей → ['symptoms', 'localization', 'duration', 'chronic_or_allergies']
  SD-1 (Universal 4) | ortho: 4 полей → ['symptoms', 'localization', 'duration', 'chronic_or_allergies']
  SD-1 (Universal 4) | therapy: 4 полей → ['symptoms', 'localization', 'duration', 'chronic_or_allergies']
  SD-2 (Type-specific) | acute_pain: 6 полей → ['chief_complaint', 'localization', 'duration', 'intensity', 'onset', 'triggers']
  SD-2 (Type-specific) | chronic_pain: 6 полей → ['chief_complaint', 'localization', 'duration', 'intensity', 'onset', 'relievers']
  SD-2 (Type-specific) | esthetics: 3 полей → ['chief_complaint', 'localization', 'desired_outcome']
  SD-2 (Type-specific) | ortho: 3 полей → ['chief_complaint', 'localizatio

In [35]:
# Валидация: все поля имеют вопросы, нет пропусков
print("=== Валидация покрытия вопросами ===")
all_ok = True
for sd_id, sd in SCHEMA_DESIGNS.items():
    questions = sd["questions"]
    for ct in COMPLAINT_TYPES:
        fields = sd["fields_fn"](ct)
        missing_q = [f for f in fields if f not in questions]
        if missing_q:
            print(f"  ⚠ {sd_id} | {ct}: нет вопросов для {missing_q}")
            all_ok = False

if all_ok:
    print("  ✓ Все поля покрыты вопросами во всех схемах")

# Сравнительная таблица размеров
print("\n=== Количество полей по схеме × тип жалобы ===")
rows = []
for ct in COMPLAINT_TYPES:
    row = {"complaint_type": ct}
    for sd_id, sd in SCHEMA_DESIGNS.items():
        row[sd_id] = len(sd["fields_fn"](ct))
    rows.append(row)

df_schema_sizes = pd.DataFrame(rows).set_index("complaint_type")
print(df_schema_sizes.to_string())

=== Валидация покрытия вопросами ===
  ✓ Все поля покрыты вопросами во всех схемах

=== Количество полей по схеме × тип жалобы ===
                SD-1  SD-2  SD-3  SD-4
complaint_type                        
acute_pain         4     6     7     8
chronic_pain       4     6     7     8
esthetics          4     3     5     8
ortho              4     3     5     8
therapy            4     3     5     8


In [36]:
# Экспорт JSON Schema (legacy, для совместимости с EXPERIMENT_PLAN.md артефактами)
from utils.schemas import export_schema
export_schema(CONFIGS_DIR / 'intake_schema.json')
print(f"JSON Schema (draft-07) экспортирована: {CONFIGS_DIR / 'intake_schema.json'}")

JSON Schema (draft-07) экспортирована: /Users/kazdoraw/developer/med-agent/study/configs/intake_schema.json


## 3. State Machine Implementation (Strategy B — Deterministic SM + Regex)

### 3.1 Стадии SM и вспомогательные функции

In [37]:
class SMStage(str, Enum):
    """Стадии State Machine для сбора анамнеза."""
    IDLE = "idle"                # Начальное состояние
    DETECTING = "detecting"      # Определение типа жалобы
    COLLECTING = "collecting"    # Сбор обязательных полей
    COMPLETE = "complete"        # Все поля заполнены
    STOPPED = "stopped"          # Прерван (user_stop / max_turns)


# --- Определение типа жалобы по ключевым словам ---
COMPLAINT_KEYWORDS: Dict[str, List[str]] = {
    "acute_pain":   ["острая", "сильно болит", "невыносим", "срочно", "резкая боль"],
    "chronic_pain": ["ноет", "периодически", "давно", "хронич", "тупая боль"],
    "esthetics":    ["красив", "отбел", "винир", "эстетик", "улыбк", "белые зубы"],
    "ortho":        ["прикус", "кривые", "брекет", "выравн", "скученность"],
    "therapy":      ["кариес", "осмотр", "профилактик", "чистк", "давно не был"],
}


def detect_complaint_type(text: str) -> str:
    """Определить тип жалобы по ключевым словам. Fallback: acute_pain."""
    text_lower = text.lower()
    for ctype, keywords in COMPLAINT_KEYWORDS.items():
        if any(kw in text_lower for kw in keywords):
            return ctype
    return "acute_pain"


def extract_field_value_regex(response: str, field_name: str) -> Optional[str]:
    """
    EM-1: Извлечение значения поля через regex/keyword matching.
    Источник: utils/llm.py:extract_field_value() — адаптировано.
    """
    response = response.strip()

    # Пустые / отрицательные ответы
    if not response or response.lower() in ("не знаю", "не помню", "нет", "-", ""):
        return None

    # intensity: число 1-10
    if field_name == "intensity":
        numbers = re.findall(r"\d+", response)
        if numbers:
            val = int(numbers[0])
            if 1 <= val <= 10:
                return str(val)
        # Текстовые описания интенсивности
        if any(w in response.lower() for w in ["сильно", "очень", "невыносим"]):
            return "8"
        if any(w in response.lower() for w in ["умеренн", "средн"]):
            return "5"
        if any(w in response.lower() for w in ["слаб", "терпим"]):
            return "3"

    # boolean-подобные поля
    if field_name in ("temperature", "swelling", "bleeding", "pregnancy"):
        if any(w in response.lower() for w in ["да", "есть", "имеется"]):
            return "да"
        if any(w in response.lower() for w in ["нет", "отсутствует", "не"]):
            return "нет"

    # Общий случай: возвращаем как есть, если длина > 1
    return response if len(response) > 1 else None


print("SMStage, detect_complaint_type, extract_field_value_regex — определены.")

SMStage, detect_complaint_type, extract_field_value_regex — определены.


In [38]:
@dataclass
class IntakeStateMachine:
    """
    State Machine для структурированного сбора анамнеза (Strategy B — CS-B).
    
    Поддерживает 4 дизайна схем (SD-1..SD-4) через параметр schema_id.
    Стадии: IDLE → DETECTING → COLLECTING → COMPLETE | STOPPED.
    Извлечение: EM-1 (regex/keyword).
    
    Атрибуты:
        schema_id: Дизайн схемы ("SD-1", "SD-2", "SD-3", "SD-4")
        max_turns: Максимум ходов (guardrail)
        cooperation_rate: Вероятность кооперативного ответа пациента
    """
    schema_id: str = "SD-1"
    max_turns: int = MAX_TURNS_SM
    cooperation_rate: float = COOPERATION_RATE_DEFAULT
    
    # Внутреннее состояние (инициализируется в __post_init__)
    stage: SMStage = field(default=SMStage.IDLE, init=False)
    complaint_type: Optional[str] = field(default=None, init=False)
    filled_fields: Dict[str, Any] = field(default_factory=dict, init=False)
    required_fields: List[str] = field(default_factory=list, init=False)
    turn_count: int = field(default=0, init=False)
    stop_reason: Optional[str] = field(default=None, init=False)
    questions_log: List[Dict[str, Any]] = field(default_factory=list, init=False)
    
    def _get_schema_config(self) -> Dict:
        """Получить конфиг текущей схемы из SCHEMA_DESIGNS."""
        return SCHEMA_DESIGNS[self.schema_id]
    
    def _get_questions_map(self) -> Dict[str, str]:
        """Получить маппинг field → question для текущей схемы."""
        return self._get_schema_config()["questions"]
    
    def initialize(self, initial_message: str) -> str:
        """
        Инициализация SM первым сообщением пациента.
        
        Returns:
            Первый вопрос ассистента.
        """
        self.stage = SMStage.DETECTING
        self.complaint_type = detect_complaint_type(initial_message)
        
        # Определяем required fields по схеме
        fields_fn = self._get_schema_config()["fields_fn"]
        self.required_fields = fields_fn(self.complaint_type)
        
        # chief_complaint / symptoms заполняем из initial_message
        first_field = self.required_fields[0]
        self.filled_fields[first_field] = initial_message
        
        self.stage = SMStage.COLLECTING
        return self._next_question()
    
    def _missing_fields(self) -> List[str]:
        """Список незаполненных обязательных полей."""
        return [f for f in self.required_fields if f not in self.filled_fields]
    
    def _next_question(self) -> Optional[str]:
        """Сформировать следующий вопрос или завершить сбор."""
        missing = self._missing_fields()
        
        if not missing:
            self.stage = SMStage.COMPLETE
            self.stop_reason = "complete"
            return None
        
        if self.turn_count >= self.max_turns:
            self.stage = SMStage.STOPPED
            self.stop_reason = "max_turns"
            return None
        
        next_field = missing[0]
        questions_map = self._get_questions_map()
        question = questions_map.get(next_field, f"Расскажите подробнее о {next_field}.")
        
        self.questions_log.append({
            "turn": self.turn_count,
            "field": next_field,
            "question": question,
        })
        return question
    
    def process_response(self, response: str) -> Optional[str]:
        """
        Обработать ответ пациента и вернуть следующий вопрос.
        
        Returns:
            Следующий вопрос или None (если SM завершён).
        """
        if self.stage != SMStage.COLLECTING:
            return None
        
        self.turn_count += 1
        
        # Проверка стоп-сигналов пациента
        if any(w in response.lower() for w in ["хватит", "достаточно", "не хочу", "стоп"]):
            self.stage = SMStage.STOPPED
            self.stop_reason = "user_stop"
            return None
        
        # Определяем целевое поле (первое незаполненное)
        missing = self._missing_fields()
        if not missing:
            self.stage = SMStage.COMPLETE
            self.stop_reason = "complete"
            return None
        
        target_field = missing[0]
        
        # EM-1: regex/keyword extraction
        value = extract_field_value_regex(response, target_field)
        if value is not None:
            self.filled_fields[target_field] = value
        
        return self._next_question()
    
    def get_completion_rate(self) -> float:
        """Доля заполненных обязательных полей (0.0 .. 1.0)."""
        if not self.required_fields:
            return 1.0
        filled = sum(1 for f in self.required_fields if f in self.filled_fields)
        return filled / len(self.required_fields)
    
    def is_expert_sufficient(self) -> bool:
        """Достаточно ли данных для первичного осмотра."""
        return self.get_completion_rate() >= EXPERT_THRESHOLD


# --- Быстрый тест ---
sm_test = IntakeStateMachine(schema_id="SD-1")
first_q = sm_test.initialize("У меня сильно болит зуб, не могу спать")
print(f"Schema: SD-1 | Тип жалобы: {sm_test.complaint_type}")
print(f"Required: {sm_test.required_fields}")
print(f"Filled: {list(sm_test.filled_fields.keys())}")
print(f"Missing: {sm_test._missing_fields()}")
print(f"Первый вопрос: {first_q}")

# Тест с SD-2
sm_test2 = IntakeStateMachine(schema_id="SD-2")
q2 = sm_test2.initialize("Хочу исправить прикус")
print(f"\nSchema: SD-2 | Тип: {sm_test2.complaint_type} | Required: {sm_test2.required_fields}")
print(f"Первый вопрос: {q2}")

Schema: SD-1 | Тип жалобы: acute_pain
Required: ['symptoms', 'localization', 'duration', 'chronic_or_allergies']
Filled: ['symptoms']
Missing: ['localization', 'duration', 'chronic_or_allergies']
Первый вопрос: Где именно болит? Укажите место.

Schema: SD-2 | Тип: ortho | Required: ['chief_complaint', 'localization', 'bite_issues']
Первый вопрос: Где именно болит или беспокоит? Укажите зуб или область.


### 3.2 Диаграмма State Machine

```
IDLE → DETECTING → COLLECTING ⟲ → COMPLETE
                       ↓
                    STOPPED (user_stop | max_turns)

Переходы:
  IDLE → DETECTING:   initialize(initial_message)
  DETECTING → COLLECTING: complaint_type определён, required_fields загружены
  COLLECTING → COLLECTING: process_response() → поле заполнено, есть missing
  COLLECTING → COMPLETE:   _missing_fields() == []
  COLLECTING → STOPPED:    turn_count >= max_turns ИЛИ user_stop
```

In [39]:
# Сохраняем ASCII-диаграмму обновлённого SM
ascii_diagram = """
+------+     initialize()     +-----------+    type detected    +------------+
| IDLE | ──────────────────→  | DETECTING | ──────────────────→ | COLLECTING |
+------+                      +-----------+                     +-----+------+
                                                                      |
                                                          ┌───────────┤
                                                          │  process_response()
                                                          │  field filled, has missing
                                                          ↓           │
                                                     +----+------+    │
                                                     | COLLECTING |←──┘
                                                     +----+------+
                                                          │
                                          ┌───────────────┼───────────────┐
                                          │               │               │
                                   missing == []    turn >= max     user_stop
                                          │               │               │
                                          ↓               ↓               ↓
                                    +----------+    +---------+    +---------+
                                    | COMPLETE |    | STOPPED |    | STOPPED |
                                    +----------+    +---------+    +---------+
"""
print(ascii_diagram)

# Сохранить в файл
diagram_path = OUTPUT_DIAGRAMS / 'd2_state_machine_v2.txt'
with open(diagram_path, 'w', encoding='utf-8') as f:
    f.write(ascii_diagram)
print(f"Диаграмма сохранена: {diagram_path}")


+------+     initialize()     +-----------+    type detected    +------------+
| IDLE | ──────────────────→  | DETECTING | ──────────────────→ | COLLECTING |
+------+                      +-----------+                     +-----+------+
                                                                      |
                                                          ┌───────────┤
                                                          │  process_response()
                                                          │  field filled, has missing
                                                          ↓           │
                                                     +----+------+    │
                                                     | COLLECTING |←──┘
                                                     +----+------+
                                                          │
                                          ┌───────────────┼───────────────┐
                                

## 4. Генерация данных

30 кейсов (6 на тип жалобы) с gold standard полями для каждого дизайна схемы.
Источник шаблонов: `utils/data.py:CASE_TEMPLATES`.

In [40]:
# === Расширенные ответы пациента для всех схем ===
# Fallback ответы когда нет patient_answers в кейсе
PATIENT_RESPONSES_EXT: Dict[str, List[str]] = {
    # SD-1 production поля
    "symptoms": [
        "Ну болит что-то, не знаю как объяснить", "Дискомфорт какой-то при еде",
        "Ноет периодически, то сильнее то слабее", "Чувствительность появилась",
    ],
    "chronic_or_allergies": [
        "Вроде ничего нет, здоров", "Что-то было, не помню точно",
        "Нет, аллергий нет", "Не знаю, давно не проверялся",
    ],
    # SD-2 поля
    "localization": ["Справа где-то внизу", "Слева наверху кажется", "Передний зуб, большой"],
    "duration": ["Ну дней пять наверное", "Давно уже, не помню точно", "Со вчера"],
    "intensity": ["Ну терпимо в целом", "Сильно болит", "Средне, на 5 из 10 примерно"],
    "onset": ["Не помню, постепенно как-то", "После еды началось", "Утром проснулся — болит"],
    "triggers": ["От холодного хуже", "Когда жую — больно", "Ночью усиливается"],
    "relievers": ["Таблетку выпью — проходит", "Ничего не помогает особо"],
    "chief_complaint": ["Ну болит зуб", "Проблема какая-то с зубами", "Дискомфорт"],
    "desired_outcome": ["Чтоб красиво было", "Ровные зубы хочу"],
    "last_visit": ["Давно не был, год наверное", "Полгода назад примерно"],
    "bite_issues": ["Прикус неправильный вроде", "Зубы кривые стоят"],
    "medications": ["Нет, ничего не принимаю", "Иногда обезболивающее пью"],
}


def simulate_patient_response(
    field: str,
    patient_answers: Dict[str, Any],
    cooperation_rate: float = COOPERATION_RATE_DEFAULT,
) -> str:
    """
    Симуляция ответа пациента на вопрос SM/LLM.
    
    ВАЖНО: принимает patient_answers (разговорная речь), НЕ gold_values.
    Gold values используются ТОЛЬКО для compute_extraction_accuracy.
    
    С вероятностью cooperation_rate возвращает patient_answer (реалистичная речь),
    иначе — случайный ответ из PATIENT_RESPONSES_EXT (шумный fallback).
    """
    # Кооперативный ответ (реалистичная речь пациента)
    if field in patient_answers and random.random() < cooperation_rate:
        return str(patient_answers[field])
    
    # Некооперативный / fallback: случайный ответ из пула
    if field in PATIENT_RESPONSES_EXT:
        return random.choice(PATIENT_RESPONSES_EXT[field])
    
    return "Не знаю, затрудняюсь ответить"


# === Маппинг gold values SD-2 → SD-1 ===
def map_gold_to_schema(gold_values: Dict[str, Any], complaint_type: str, schema_id: str) -> Dict[str, Any]:
    """
    Маппинг gold standard values из SD-2 формата в формат целевой схемы.
    
    SD-2 gold → SD-1: chief_complaint → symptoms, allergies+chronic → chronic_or_allergies
    """
    mapped = gold_values.copy()
    
    if schema_id in ("SD-1", "SD-3", "SD-4"):
        # chief_complaint → symptoms
        if "chief_complaint" in mapped and "symptoms" not in mapped:
            mapped["symptoms"] = mapped["chief_complaint"]
        # Объединяем аллергии и хронические
        if "chronic_or_allergies" not in mapped:
            parts = []
            if "allergies" in mapped:
                parts.append(str(mapped["allergies"]))
            if "chronic_conditions" in mapped:
                parts.append(str(mapped["chronic_conditions"]))
            mapped["chronic_or_allergies"] = ", ".join(parts) if parts else "нет"
        # medications fallback
        if schema_id == "SD-4" and "medications" not in mapped:
            mapped["medications"] = "не указано"
    
    return mapped


# === Загрузка / генерация кейсов ===
DATA_PATH_V2 = DATA_DIR / 'd2_cases_v2.jsonl'

# Генерируем свежие кейсы (seed гарантирует воспроизводимость)
cases = generate_d2_cases(n=N_CASES, seed=SEED)

# Сохраняем JSONL
with open(DATA_PATH_V2, 'w', encoding='utf-8') as f:
    for case in cases:
        f.write(json.dumps(case, ensure_ascii=False) + '\n')

print(f"Кейсов: {len(cases)} | Сохранено: {DATA_PATH_V2}")
print(f"\nРаспределение по типам жалоб:")
type_counts = {}
for case in cases:
    ct = case["target_complaint_type"]
    type_counts[ct] = type_counts.get(ct, 0) + 1
for ct, cnt in sorted(type_counts.items()):
    print(f"  {ct}: {cnt}")

# Показать примеры: patient_answers vs gold_values
print("\n--- Примеры: patient_answers vs gold_values ---")
for case in cases[:3]:
    gold_sd1 = map_gold_to_schema(case["gold_values"], case["target_complaint_type"], "SD-1")
    pa = case.get("patient_answers", {})
    print(f"\nCase {case['case_id']} ({case['target_complaint_type']})")
    print(f"  Жалоба: {case['initial_user_message']}")
    for fld in SD1_FIELDS:
        g = gold_sd1.get(fld, "—")
        p = pa.get(fld, "—")
        marker = "≠" if str(g).lower() != str(p).lower() else "="
        print(f"  {fld}: gold='{g}' {marker} patient='{p}'")

Кейсов: 10 | Сохранено: /Users/kazdoraw/developer/med-agent/study/data/d2_cases_v2.jsonl

Распределение по типам жалоб:
  acute_pain: 2
  chronic_pain: 2
  esthetics: 2
  ortho: 2
  therapy: 2

--- Примеры: patient_answers vs gold_values ---

Case 5 (esthetics)
  Жалоба: Хочу поставить виниры
  symptoms: gold='Установка виниров' ≠ patient='Хочу ровные красивые зубы, как у звёзд, чтобы идеально было'
  localization: gold='Зона улыбки' ≠ patient='Ну зону улыбки, верхние передние штук шесть-восемь наверное'
  duration: gold='—' ≠ patient='Давно хочу, наконец решился, деньги накопил'
  chronic_or_allergies: gold='Нет аллергий' ≠ patient='Нет аллергий, ничем серьёзным не болел'

Case 4 (chronic_pain)
  Жалоба: Периодически болит один и тот же зуб
  symptoms: gold='Периодическая боль' ≠ patient='То болит, то не болит, непонятно, зуб как будто живёт своей жизнью'
  localization: gold='Верхний передний зуб' ≠ patient='Передний сверху, большой такой, ну резец наверное называется'
  duration: go

In [41]:
# Сохраняем кейсы в JSONL
with open(DATA_PATH_V2, 'w', encoding='utf-8') as f:
    for case in cases:
        f.write(json.dumps(case, ensure_ascii=False) + '\n')
print(f"Кейсы сохранены: {DATA_PATH_V2}")

Кейсы сохранены: /Users/kazdoraw/developer/med-agent/study/data/d2_cases_v2.jsonl


## 5. Strategy B — Deterministic SM + Regex (CS-B × EM-1)

Запускаем `IntakeStateMachine` для всех 4 дизайнов схем (SD-1..SD-4).
Пациент-симулятор: `simulate_patient_response()` с `cooperation_rate=0.9`.

In [42]:
def run_strategy_b(
    case: Dict[str, Any],
    schema_id: str = "SD-1",
    cooperation_rate: float = COOPERATION_RATE_DEFAULT,
    max_turns: int = MAX_TURNS_SM,
) -> Dict[str, Any]:
    """
    Strategy B: Deterministic SM + Regex extraction (CS-B × EM-1).
    
    SM задаёт фиксированные вопросы по порядку missing fields,
    пациент-симулятор отвечает с заданной cooperation_rate.
    
    Пациент-симулятор использует patient_answers (разговорная речь),
    gold_values — ТОЛЬКО для оценки.
    """
    sm = IntakeStateMachine(
        schema_id=schema_id,
        max_turns=max_turns,
        cooperation_rate=cooperation_rate,
    )
    
    # Gold values — для ОЦЕНКИ (clinical ground truth), маппим под целевую схему
    gold_values = map_gold_to_schema(
        case.get("gold_values", {}),
        case["target_complaint_type"],
        schema_id,
    )
    # Patient answers — для СИМУЛЯТОРА (разговорная речь)
    patient_answers = case.get("patient_answers", gold_values)
    
    # Диалог
    dialog: List[Dict[str, str]] = [
        {"role": "patient", "text": case["initial_user_message"]}
    ]
    
    # Инициализация SM первым сообщением
    first_question = sm.initialize(case["initial_user_message"])
    
    if first_question:
        dialog.append({"role": "assistant", "text": first_question})
    
    # Цикл сбора
    while sm.stage == SMStage.COLLECTING:
        # Определяем текущее поле (первое missing)
        missing = sm._missing_fields()
        if not missing:
            break
        current_field = missing[0]
        
        # Пациент отвечает РАЗГОВОРНОЙ речью, не gold values
        response = simulate_patient_response(current_field, patient_answers, cooperation_rate)
        dialog.append({"role": "patient", "text": response})
        
        # Обработка SM
        next_q = sm.process_response(response)
        if next_q:
            dialog.append({"role": "assistant", "text": next_q})
    
    return {
        "case_id": case["case_id"],
        "schema_id": schema_id,
        "strategy": "B1",
        "complaint_type": sm.complaint_type,
        "target_complaint_type": case["target_complaint_type"],
        "filled_fields": sm.filled_fields.copy(),
        "required_fields": sm.required_fields.copy(),
        "turns": sm.turn_count,
        "completion_rate": sm.get_completion_rate(),
        "expert_sufficient": sm.is_expert_sufficient(),
        "stop_reason": sm.stop_reason,
        "stage": sm.stage.value,
        "dialog": dialog,
        "questions_log": sm.questions_log.copy(),
        "cooperation_rate": cooperation_rate,
    }


print("run_strategy_b() — patient_answers для симулятора, gold_values для оценки.")

run_strategy_b() — patient_answers для симулятора, gold_values для оценки.


In [43]:
# === Запуск Strategy B для всех 4 дизайнов схем ===
results_b: Dict[str, List[Dict]] = {}

for sd_id in SCHEMA_DESIGNS:
    results_b[sd_id] = []
    for case in cases:
        result = run_strategy_b(case, schema_id=sd_id)
        results_b[sd_id].append(result)

# Сводка
print("=== Strategy B (CS-B × EM-1) — Результаты по схемам ===\n")
print(f"{'Schema':<10} {'Avg Completion':>16} {'Avg Turns':>12} {'Expert Suff.':>14} {'Complete':>10}")
print("-" * 65)

for sd_id in SCHEMA_DESIGNS:
    completions = [r["completion_rate"] for r in results_b[sd_id]]
    turns = [r["turns"] for r in results_b[sd_id]]
    expert = [r["expert_sufficient"] for r in results_b[sd_id]]
    
    print(f"{sd_id:<10} {np.mean(completions):>15.1%} {np.mean(turns):>12.1f} "
          f"{np.mean(expert):>13.1%} {sum(1 for r in results_b[sd_id] if r['stop_reason']=='complete'):>10}/{N_CASES}")

=== Strategy B (CS-B × EM-1) — Результаты по схемам ===

Schema       Avg Completion    Avg Turns   Expert Suff.   Complete
-----------------------------------------------------------------
SD-1                 95.0%          2.9         90.0%          9/10
SD-2                100.0%          3.5        100.0%         10/10
SD-3                 94.0%          4.8         90.0%          9/10
SD-4                 92.5%          6.5         90.0%          9/10


In [44]:
# === Детальный анализ по типам жалоб для SD-1 (primary) ===
print("=== Strategy B × SD-1 — Детализация по типам жалоб ===\n")
for ct in COMPLAINT_TYPES:
    ct_results = [r for r in results_b["SD-1"] if r["target_complaint_type"] == ct]
    if not ct_results:
        continue
    completions = [r["completion_rate"] for r in ct_results]
    turns = [r["turns"] for r in ct_results]
    print(f"{ct}: completion={np.mean(completions):.0%}, turns={np.mean(turns):.1f}, n={len(ct_results)}")

# Per-field completion для SD-1
print("\n=== Per-field completion (SD-1) ===")
for field_name in SD1_FIELDS:
    filled = sum(1 for r in results_b["SD-1"] if field_name in r["filled_fields"])
    total = len(results_b["SD-1"])
    print(f"  {field_name}: {filled}/{total} ({filled/total:.0%})")

=== Strategy B × SD-1 — Детализация по типам жалоб ===

acute_pain: completion=100%, turns=3.0, n=2
chronic_pain: completion=100%, turns=3.0, n=2
esthetics: completion=100%, turns=3.0, n=2
ortho: completion=75%, turns=2.5, n=2
therapy: completion=100%, turns=3.0, n=2

=== Per-field completion (SD-1) ===
  symptoms: 10/10 (100%)
  localization: 10/10 (100%)
  duration: 9/10 (90%)
  chronic_or_allergies: 9/10 (90%)


In [45]:
# === Пример диалога Strategy B × SD-1 ===
example = results_b["SD-1"][0]
print(f"Case {example['case_id']} | {example['target_complaint_type']} | SD-1")
print(f"Completion: {example['completion_rate']:.0%} | Turns: {example['turns']} | Stop: {example['stop_reason']}")
print(f"Required: {example['required_fields']}")
print(f"Filled: {list(example['filled_fields'].keys())}")
print("\nДиалог:")
for msg in example["dialog"]:
    role_icon = "👤" if msg["role"] == "patient" else "🤖"
    text = msg["text"][:100] + "..." if len(msg["text"]) > 100 else msg["text"]
    print(f"  {role_icon} {text}")

Case 5 | esthetics | SD-1
Completion: 100% | Turns: 3 | Stop: complete
Required: ['symptoms', 'localization', 'duration', 'chronic_or_allergies']
Filled: ['symptoms', 'localization', 'duration', 'chronic_or_allergies']

Диалог:
  👤 Хочу поставить виниры
  🤖 Где именно болит? Укажите место.
  👤 Ну зону улыбки, верхние передние штук шесть-восемь наверное
  🤖 Как давно это беспокоит?
  👤 Давно хочу, наконец решился, деньги накопил
  🤖 Есть ли у вас хронические заболевания или аллергии?
  👤 Нет аллергий, ничем серьёзным не болел


## 6. Формальная теорема полноты SM

**Теорема 1 (Гарантия полноты).**
Пусть SM имеет |R| обязательных полей, max_turns ходов, а пациент отвечает кооперативно с вероятностью p.
Тогда вероятность заполнения ВСЕХ полей:

$$P(\text{complete}) \geq 1 - |R| \cdot (1 - p)^{\lfloor \text{max\_turns} / |R| \rfloor}$$

**Доказательство:**
- SM задаёт вопрос про конкретное поле; при кооперативном ответе поле заполняется
- На каждое поле приходится ≥ ⌊max_turns / |R|⌋ попыток
- Вероятность НЕ заполнить одно поле за k попыток: $(1-p)^k$
- По union bound: $P(\text{incomplete}) \leq |R| \cdot (1-p)^k$

**Следствие:** При p=0.9, |R|=4, max_turns=8 → k=2 → P(complete) ≥ 1 - 4·0.01 = **96%**

In [46]:
# === Визуализация теоремы: теоретическая кривая P(complete) vs cooperation_rate ===
def theoretical_completion_prob(p: float, n_fields: int, max_turns: int) -> float:
    """Нижняя граница P(complete) по Теореме 1."""
    k = max_turns // n_fields  # попыток на поле
    if k == 0:
        return 0.0
    return max(0.0, 1.0 - n_fields * (1 - p) ** k)


# Построение кривых для разных схем
coop_rates = np.linspace(0.3, 1.0, 50)

fig, ax = plt.subplots(figsize=(10, 6))

for sd_id in ["SD-1", "SD-4", "SD-2"]:
    # Берём среднее кол-во полей по типам
    n_fields_list = [len(SCHEMA_DESIGNS[sd_id]["fields_fn"](ct)) for ct in COMPLAINT_TYPES]
    n_fields_avg = int(np.mean(n_fields_list))
    
    probs = [theoretical_completion_prob(p, n_fields_avg, MAX_TURNS_SM) for p in coop_rates]
    ax.plot(coop_rates, probs, label=f"{sd_id} (|R|≈{n_fields_avg})", linewidth=2)

ax.axhline(y=0.9, color='green', linestyle='--', alpha=0.7, label='Target 90%')
ax.axvline(x=0.9, color='gray', linestyle=':', alpha=0.5, label='p=0.9')
ax.set_xlabel('Cooperation Rate (p)')
ax.set_ylabel('P(complete) — нижняя граница')
ax.set_title('Теорема 1: Гарантия полноты SM по дизайнам схем')
ax.legend()
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)

plt.tight_layout()
fig_path = OUTPUT_FIGURES / 'd2_theorem_completion.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Сохранено: {fig_path}")

# Числовые значения для ключевых точек
print("\n=== Ключевые значения P(complete) ===")
for sd_id in SCHEMA_DESIGNS:
    for ct in COMPLAINT_TYPES:
        n_f = len(SCHEMA_DESIGNS[sd_id]["fields_fn"](ct))
        p_val = theoretical_completion_prob(0.9, n_f, MAX_TURNS_SM)
        print(f"  {sd_id} × {ct}: |R|={n_f}, P(complete|p=0.9) ≥ {p_val:.4f}")

Сохранено: /Users/kazdoraw/developer/med-agent/study/outputs/figures/d2_theorem_completion.png

=== Ключевые значения P(complete) ===
  SD-1 × acute_pain: |R|=4, P(complete|p=0.9) ≥ 0.9600
  SD-1 × chronic_pain: |R|=4, P(complete|p=0.9) ≥ 0.9600
  SD-1 × esthetics: |R|=4, P(complete|p=0.9) ≥ 0.9600
  SD-1 × ortho: |R|=4, P(complete|p=0.9) ≥ 0.9600
  SD-1 × therapy: |R|=4, P(complete|p=0.9) ≥ 0.9600
  SD-2 × acute_pain: |R|=6, P(complete|p=0.9) ≥ 0.4000
  SD-2 × chronic_pain: |R|=6, P(complete|p=0.9) ≥ 0.4000
  SD-2 × esthetics: |R|=3, P(complete|p=0.9) ≥ 0.9700
  SD-2 × ortho: |R|=3, P(complete|p=0.9) ≥ 0.9700
  SD-2 × therapy: |R|=3, P(complete|p=0.9) ≥ 0.9700
  SD-3 × acute_pain: |R|=7, P(complete|p=0.9) ≥ 0.3000
  SD-3 × chronic_pain: |R|=7, P(complete|p=0.9) ≥ 0.3000
  SD-3 × esthetics: |R|=5, P(complete|p=0.9) ≥ 0.5000
  SD-3 × ortho: |R|=5, P(complete|p=0.9) ≥ 0.5000
  SD-3 × therapy: |R|=5, P(complete|p=0.9) ≥ 0.5000
  SD-4 × acute_pain: |R|=8, P(complete|p=0.9) ≥ 0.2000
  SD-4 

/var/folders/6w/j5zxb2n50dj_hrb7t96x08bh0000gn/T/ipykernel_87878/3265679980.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Monte Carlo Sensitivity Analysis

Исследуем чувствительность SM к cooperation_rate пациента.
- `cooperation_rates = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]`
- `n_simulations = 1000` на каждую точку
- Без API вызовов (детерминистическая симуляция)
- Bootstrap 95% CI

In [47]:
# === Monte Carlo: completion rate vs cooperation rate ===
COOP_RATES = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
MC_SCHEMAS = ["SD-1", "SD-2"]  # Сравниваем production vs type-specific

mc_results: Dict[str, Dict[float, List[float]]] = {}

for sd_id in MC_SCHEMAS:
    mc_results[sd_id] = {}
    for coop in COOP_RATES:
        completions = []
        for _ in range(N_MONTE_CARLO):
            # Выбираем случайный кейс
            case = random.choice(cases)
            result = run_strategy_b(case, schema_id=sd_id, cooperation_rate=coop)
            completions.append(result["completion_rate"])
        mc_results[sd_id][coop] = completions

print(f"Monte Carlo завершён: {len(MC_SCHEMAS)} схем × {len(COOP_RATES)} rates × {N_MONTE_CARLO} симуляций")

# Сводка
print(f"\n{'Schema':<8} {'Coop':>6} {'Mean':>8} {'Std':>8} {'CI_lo':>8} {'CI_hi':>8}")
print("-" * 50)
for sd_id in MC_SCHEMAS:
    for coop in COOP_RATES:
        vals = mc_results[sd_id][coop]
        mean = np.mean(vals)
        std = np.std(vals)
        ci_lo = np.percentile(vals, 2.5)
        ci_hi = np.percentile(vals, 97.5)
        print(f"{sd_id:<8} {coop:>6.1f} {mean:>8.3f} {std:>8.3f} {ci_lo:>8.3f} {ci_hi:>8.3f}")

Monte Carlo завершён: 2 схем × 6 rates × 1000 симуляций

Schema     Coop     Mean      Std    CI_lo    CI_hi
--------------------------------------------------
SD-1        0.5    0.972    0.115    0.500    1.000
SD-1        0.6    0.965    0.128    0.500    1.000
SD-1        0.7    0.962    0.132    0.500    1.000
SD-1        0.8    0.966    0.126    0.500    1.000
SD-1        0.9    0.950    0.151    0.500    1.000
SD-1        1.0    0.936    0.166    0.500    1.000
SD-2        0.5    1.000    0.000    1.000    1.000
SD-2        0.6    1.000    0.000    1.000    1.000
SD-2        0.7    1.000    0.000    1.000    1.000
SD-2        0.8    1.000    0.000    1.000    1.000
SD-2        0.9    1.000    0.000    1.000    1.000
SD-2        1.0    1.000    0.000    1.000    1.000


In [48]:
# === Визуализация Monte Carlo: Cooperation Rate vs Completion ===
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- Левый: линейный график с CI bands ---
ax1 = axes[0]
colors = {"SD-1": "#1f77b4", "SD-2": "#ff7f0e"}

for sd_id in MC_SCHEMAS:
    means = [np.mean(mc_results[sd_id][c]) for c in COOP_RATES]
    ci_lo = [np.percentile(mc_results[sd_id][c], 2.5) for c in COOP_RATES]
    ci_hi = [np.percentile(mc_results[sd_id][c], 97.5) for c in COOP_RATES]
    
    ax1.plot(COOP_RATES, means, 'o-', label=f"{sd_id} ({SCHEMA_DESIGNS[sd_id]['name']})",
             color=colors[sd_id], linewidth=2)
    ax1.fill_between(COOP_RATES, ci_lo, ci_hi, alpha=0.2, color=colors[sd_id])

# Теоретическая граница для SD-1
n_f_sd1 = len(SD1_FIELDS)
theo = [theoretical_completion_prob(p, n_f_sd1, MAX_TURNS_SM) for p in COOP_RATES]
ax1.plot(COOP_RATES, theo, '--', color='gray', alpha=0.7, label='Теорема 1 (SD-1)')

ax1.axhline(y=0.9, color='green', linestyle=':', alpha=0.5, label='Target 90%')
ax1.set_xlabel('Cooperation Rate')
ax1.set_ylabel('Avg Completion Rate')
ax1.set_title('Fig 7: Cooperation Rate vs Completion (Monte Carlo)')
ax1.legend(fontsize=9)
ax1.set_ylim(0, 1.05)
ax1.grid(True, alpha=0.3)

# --- Правый: boxplot для p=0.7 и p=0.9 ---
ax2 = axes[1]
box_data = []
box_labels = []
for sd_id in MC_SCHEMAS:
    for coop in [0.7, 0.9]:
        box_data.append(mc_results[sd_id][coop])
        box_labels.append(f"{sd_id}\np={coop}")

bp = ax2.boxplot(box_data, labels=box_labels, patch_artist=True)
box_colors = [colors["SD-1"], colors["SD-1"], colors["SD-2"], colors["SD-2"]]
for patch, color in zip(bp['boxes'], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)

ax2.axhline(y=0.9, color='green', linestyle=':', alpha=0.5)
ax2.set_ylabel('Completion Rate')
ax2.set_title('Распределение completion при p=0.7 и p=0.9')
ax2.set_ylim(0, 1.1)

plt.tight_layout()
fig_path = OUTPUT_FIGURES / 'd2_monte_carlo_cooperation.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Сохранено: {fig_path}")

/var/folders/6w/j5zxb2n50dj_hrb7t96x08bh0000gn/T/ipykernel_87878/2049524673.py:39: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax2.boxplot(box_data, labels=box_labels, patch_artist=True)


Сохранено: /Users/kazdoraw/developer/med-agent/study/outputs/figures/d2_monte_carlo_cooperation.png


/var/folders/6w/j5zxb2n50dj_hrb7t96x08bh0000gn/T/ipykernel_87878/2049524673.py:53: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Drop-off Analysis

Моделируем сценарий, когда пациент прерывает диалог с вероятностью `drop_rate` на каждом ходу.
- `drop_rates = [0.02, 0.05, 0.10, 0.15]`
- `n_simulations = 1000` на точку

In [49]:
# === Drop-off: SM с вероятностью прерывания пациентом ===
DROP_RATES = [0.02, 0.05, 0.10, 0.15]

def run_strategy_b_with_dropout(
    case: Dict[str, Any],
    schema_id: str,
    cooperation_rate: float,
    drop_rate: float,
) -> Dict[str, Any]:
    """Strategy B с добавлением drop-off (пациент уходит на каждом ходу с P=drop_rate)."""
    sm = IntakeStateMachine(schema_id=schema_id, cooperation_rate=cooperation_rate)
    gold_values = map_gold_to_schema(case.get("gold_values", {}), case["target_complaint_type"], schema_id)
    
    first_q = sm.initialize(case["initial_user_message"])
    
    while sm.stage == SMStage.COLLECTING:
        # Drop-off check
        if random.random() < drop_rate:
            sm.stage = SMStage.STOPPED
            sm.stop_reason = "user_drop"
            break
        
        missing = sm._missing_fields()
        if not missing:
            break
        
        response = simulate_patient_response(missing[0], gold_values, cooperation_rate)
        sm.process_response(response)
    
    return {
        "completion_rate": sm.get_completion_rate(),
        "turns": sm.turn_count,
        "stop_reason": sm.stop_reason,
    }


# Запуск Monte Carlo для drop-off
drop_results: Dict[str, Dict[float, List[float]]] = {}

for sd_id in MC_SCHEMAS:
    drop_results[sd_id] = {}
    for dr in DROP_RATES:
        completions = []
        for _ in range(N_MONTE_CARLO):
            case = random.choice(cases)
            r = run_strategy_b_with_dropout(case, sd_id, COOPERATION_RATE_DEFAULT, dr)
            completions.append(r["completion_rate"])
        drop_results[sd_id][dr] = completions

print(f"Drop-off analysis завершён: {len(MC_SCHEMAS)} схем × {len(DROP_RATES)} rates × {N_MONTE_CARLO}")

# Сводка
print(f"\n{'Schema':<8} {'Drop':>6} {'Mean':>8} {'CI_lo':>8} {'CI_hi':>8}")
print("-" * 42)
for sd_id in MC_SCHEMAS:
    for dr in DROP_RATES:
        vals = drop_results[sd_id][dr]
        print(f"{sd_id:<8} {dr:>6.2f} {np.mean(vals):>8.3f} "
              f"{np.percentile(vals, 2.5):>8.3f} {np.percentile(vals, 97.5):>8.3f}")

Drop-off analysis завершён: 2 схем × 4 rates × 1000

Schema     Drop     Mean    CI_lo    CI_hi
------------------------------------------
SD-1       0.02    0.917    0.500    1.000
SD-1       0.05    0.878    0.250    1.000
SD-1       0.10    0.816    0.250    1.000
SD-1       0.15    0.766    0.250    1.000
SD-2       0.02    0.964    0.333    1.000
SD-2       0.05    0.915    0.333    1.000
SD-2       0.10    0.839    0.167    1.000
SD-2       0.15    0.769    0.167    1.000


In [50]:
# === Визуализация Drop-off: Fig 8 ===
fig, ax = plt.subplots(figsize=(10, 6))

colors = {"SD-1": "#1f77b4", "SD-2": "#ff7f0e"}

for sd_id in MC_SCHEMAS:
    means = [np.mean(drop_results[sd_id][dr]) for dr in DROP_RATES]
    ci_lo = [np.percentile(drop_results[sd_id][dr], 2.5) for dr in DROP_RATES]
    ci_hi = [np.percentile(drop_results[sd_id][dr], 97.5) for dr in DROP_RATES]
    
    ax.plot(DROP_RATES, means, 'o-', label=f"{sd_id} ({SCHEMA_DESIGNS[sd_id]['name']})",
            color=colors[sd_id], linewidth=2)
    ax.fill_between(DROP_RATES, ci_lo, ci_hi, alpha=0.2, color=colors[sd_id])

ax.axhline(y=0.9, color='green', linestyle=':', alpha=0.5, label='Target 90%')
ax.set_xlabel('Drop-off Rate (per turn)')
ax.set_ylabel('Avg Completion Rate')
ax.set_title('Fig 8: Drop-off Rate vs Completion (Monte Carlo, p=0.9)')
ax.legend()
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)

plt.tight_layout()
fig_path = OUTPUT_FIGURES / 'd2_drop_off_analysis.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Сохранено: {fig_path}")

Сохранено: /Users/kazdoraw/developer/med-agent/study/outputs/figures/d2_drop_off_analysis.png


/var/folders/6w/j5zxb2n50dj_hrb7t96x08bh0000gn/T/ipykernel_87878/1511584150.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Визуализации Strategy B

Completion distribution, per-field heatmap, schema comparison.

In [51]:
# === Completion distribution boxplot: SD-1 vs SD-2 vs SD-3 vs SD-4 ===
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- Левый: boxplot по схемам ---
ax1 = axes[0]
box_data = [
    [r["completion_rate"] for r in results_b[sd_id]]
    for sd_id in SCHEMA_DESIGNS
]
box_labels = [f"{sd_id}\n({SCHEMA_DESIGNS[sd_id]['name']})" for sd_id in SCHEMA_DESIGNS]

bp = ax1.boxplot(box_data, tick_labels=box_labels, patch_artist=True)
schema_colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]
for patch, color in zip(bp['boxes'], schema_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)

ax1.axhline(y=0.9, color='green', linestyle='--', alpha=0.7, label='Target 90%')
ax1.set_ylabel('Completion Rate')
ax1.set_title('Strategy B: Completion по дизайнам схем')
ax1.set_ylim(0, 1.1)
ax1.legend()

# --- Правый: turns distribution ---
ax2 = axes[1]
turns_data = [
    [r["turns"] for r in results_b[sd_id]]
    for sd_id in SCHEMA_DESIGNS
]
bp2 = ax2.boxplot(turns_data, tick_labels=box_labels, patch_artist=True)
for patch, color in zip(bp2['boxes'], schema_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)

ax2.set_ylabel('Turns')
ax2.set_title('Strategy B: Ходы по дизайнам схем')

plt.tight_layout()
fig_path = OUTPUT_FIGURES / 'd2_strategy_b_comparison.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Сохранено: {fig_path}")

Сохранено: /Users/kazdoraw/developer/med-agent/study/outputs/figures/d2_strategy_b_comparison.png


/var/folders/6w/j5zxb2n50dj_hrb7t96x08bh0000gn/T/ipykernel_87878/2642171954.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [52]:
# === Per-field completion heatmap: схемы × поля ===
# Собираем все уникальные поля по всем схемам
all_fields_set: set = set()
for sd_id in SCHEMA_DESIGNS:
    for ct in COMPLAINT_TYPES:
        all_fields_set.update(SCHEMA_DESIGNS[sd_id]["fields_fn"](ct))

all_fields_sorted = sorted(all_fields_set)

# Матрица: строки = схемы, столбцы = поля, значения = % заполнения
heatmap_data = []
for sd_id in SCHEMA_DESIGNS:
    row = {}
    for field_name in all_fields_sorted:
        # Считаем только кейсы, где это поле required
        relevant = [r for r in results_b[sd_id] if field_name in r["required_fields"]]
        if relevant:
            filled = sum(1 for r in relevant if field_name in r["filled_fields"])
            row[field_name] = filled / len(relevant)
        else:
            row[field_name] = np.nan  # Поле не required для этой схемы
    heatmap_data.append(row)

df_heatmap = pd.DataFrame(heatmap_data, index=list(SCHEMA_DESIGNS.keys()))

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(
    df_heatmap, annot=True, fmt=".0%", cmap="RdYlGn", vmin=0, vmax=1,
    linewidths=0.5, ax=ax, mask=df_heatmap.isna(),
    cbar_kws={"label": "Completion Rate"},
)
ax.set_title("Per-field Completion Rate: Schema × Field (Strategy B)")
ax.set_ylabel("Schema Design")
ax.set_xlabel("Field")
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
fig_path = OUTPUT_FIGURES / 'd2_per_field_heatmap.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Сохранено: {fig_path}")

Сохранено: /Users/kazdoraw/developer/med-agent/study/outputs/figures/d2_per_field_heatmap.png


/var/folders/6w/j5zxb2n50dj_hrb7t96x08bh0000gn/T/ipykernel_87878/2596856825.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 10. Экспорт артефактов Phase 1

In [53]:
# === Export: Metrics Summary CSV ===
rows = []
for sd_id in SCHEMA_DESIGNS:
    completions = [r["completion_rate"] for r in results_b[sd_id]]
    turns = [r["turns"] for r in results_b[sd_id]]
    expert = [r["expert_sufficient"] for r in results_b[sd_id]]
    complete_count = sum(1 for r in results_b[sd_id] if r["stop_reason"] == "complete")
    
    rows.append({
        "schema": sd_id,
        "schema_name": SCHEMA_DESIGNS[sd_id]["name"],
        "strategy": "B1 (Deterministic SM + Regex)",
        "n_cases": N_CASES,
        "avg_completion_rate": np.mean(completions),
        "std_completion_rate": np.std(completions),
        "avg_turns": np.mean(turns),
        "std_turns": np.std(turns),
        "expert_sufficient_rate": np.mean(expert),
        "complete_count": complete_count,
        "cooperation_rate": COOPERATION_RATE_DEFAULT,
    })

df_metrics = pd.DataFrame(rows)
metrics_path = OUTPUT_TABLES / 'd2_strategy_b_metrics.csv'
df_metrics.to_csv(metrics_path, index=False)
print(f"Метрики сохранены: {metrics_path}")
print(df_metrics.to_string(index=False))

Метрики сохранены: /Users/kazdoraw/developer/med-agent/study/outputs/tables/d2_strategy_b_metrics.csv
schema     schema_name                      strategy  n_cases  avg_completion_rate  std_completion_rate  avg_turns  std_turns  expert_sufficient_rate  complete_count  cooperation_rate
  SD-1     Universal 4 B1 (Deterministic SM + Regex)       10                0.950                0.150        2.9    0.30000                     0.9               9               0.9
  SD-2   Type-specific B1 (Deterministic SM + Regex)       10                1.000                0.000        3.5    1.50000                     1.0              10               0.9
  SD-3        Adaptive B1 (Deterministic SM + Regex)       10                0.940                0.180        4.8    1.32665                     0.9               9               0.9
  SD-4 Flat Extended 8 B1 (Deterministic SM + Regex)       10                0.925                0.225        6.5    1.50000                     0.9             

In [54]:
# === Export: Monte Carlo + Drop-off CSV ===
mc_rows = []
for sd_id in MC_SCHEMAS:
    for coop in COOP_RATES:
        vals = mc_results[sd_id][coop]
        mc_rows.append({
            "schema": sd_id, "analysis": "cooperation",
            "rate": coop, "mean_completion": np.mean(vals),
            "std": np.std(vals),
            "ci_lo": np.percentile(vals, 2.5),
            "ci_hi": np.percentile(vals, 97.5),
        })
    for dr in DROP_RATES:
        vals = drop_results[sd_id][dr]
        mc_rows.append({
            "schema": sd_id, "analysis": "dropout",
            "rate": dr, "mean_completion": np.mean(vals),
            "std": np.std(vals),
            "ci_lo": np.percentile(vals, 2.5),
            "ci_hi": np.percentile(vals, 97.5),
        })

df_mc = pd.DataFrame(mc_rows)
mc_path = OUTPUT_TABLES / 'd2_monte_carlo_results.csv'
df_mc.to_csv(mc_path, index=False)
print(f"Monte Carlo сохранено: {mc_path}")
print(f"Строк: {len(df_mc)}")

Monte Carlo сохранено: /Users/kazdoraw/developer/med-agent/study/outputs/tables/d2_monte_carlo_results.csv
Строк: 20


In [55]:
# === Export: Per-field completion CSV ===
field_rows = []
for sd_id in SCHEMA_DESIGNS:
    for field_name in all_fields_sorted:
        relevant = [r for r in results_b[sd_id] if field_name in r["required_fields"]]
        if relevant:
            filled = sum(1 for r in relevant if field_name in r["filled_fields"])
            field_rows.append({
                "schema": sd_id,
                "field": field_name,
                "n_required": len(relevant),
                "n_filled": filled,
                "completion_rate": filled / len(relevant),
            })

df_fields = pd.DataFrame(field_rows)
fields_path = OUTPUT_TABLES / 'd2_per_field_completion_v2.csv'
df_fields.to_csv(fields_path, index=False)
print(f"Per-field completion сохранено: {fields_path}")
print(f"Строк: {len(df_fields)}")

Per-field completion сохранено: /Users/kazdoraw/developer/med-agent/study/outputs/tables/d2_per_field_completion_v2.csv
Строк: 33


In [56]:
# === Итоговый отчёт Phase 1 ===
report = f"""# D2 Phase 1: Strategy B (Deterministic SM + Regex) — Результаты

**Дата:** {datetime.now().strftime('%Y-%m-%d %H:%M')}
**Seed:** {SEED} | **Cases:** {N_CASES} | **Max turns:** {MAX_TURNS_SM}

## Strategy B Results (CS-B × EM-1)

| Schema | Avg Completion | Avg Turns | Expert Sufficient | Complete |
|--------|---------------|-----------|-------------------|----------|
"""

for sd_id in SCHEMA_DESIGNS:
    completions = [r["completion_rate"] for r in results_b[sd_id]]
    turns = [r["turns"] for r in results_b[sd_id]]
    expert = [r["expert_sufficient"] for r in results_b[sd_id]]
    complete_n = sum(1 for r in results_b[sd_id] if r["stop_reason"] == "complete")
    report += f"| {sd_id} ({SCHEMA_DESIGNS[sd_id]['name']}) | {np.mean(completions):.1%} | {np.mean(turns):.1f} | {np.mean(expert):.1%} | {complete_n}/{N_CASES} |\n"

report += f"""
## Формальная теорема полноты
При p=0.9, |R|=4 (SD-1), max_turns={MAX_TURNS_SM}:
- Теоретическая нижняя граница: P(complete) ≥ {theoretical_completion_prob(0.9, 4, MAX_TURNS_SM):.4f}

## Monte Carlo Sensitivity (n={N_MONTE_CARLO})
- SD-1 при p=0.7: completion = {np.mean(mc_results['SD-1'][0.7]):.3f} [{np.percentile(mc_results['SD-1'][0.7], 2.5):.3f}, {np.percentile(mc_results['SD-1'][0.7], 97.5):.3f}]
- SD-1 при p=0.9: completion = {np.mean(mc_results['SD-1'][0.9]):.3f} [{np.percentile(mc_results['SD-1'][0.9], 2.5):.3f}, {np.percentile(mc_results['SD-1'][0.9], 97.5):.3f}]

## Drop-off Analysis
- SD-1 при drop=0.05: completion = {np.mean(drop_results['SD-1'][0.05]):.3f}
- SD-1 при drop=0.15: completion = {np.mean(drop_results['SD-1'][0.15]):.3f}

## Артефакты Phase 1
- `{metrics_path}`
- `{mc_path}`
- `{fields_path}`
- `{OUTPUT_FIGURES / 'd2_theorem_completion.png'}`
- `{OUTPUT_FIGURES / 'd2_monte_carlo_cooperation.png'}`
- `{OUTPUT_FIGURES / 'd2_drop_off_analysis.png'}`
- `{OUTPUT_FIGURES / 'd2_strategy_b_comparison.png'}`
- `{OUTPUT_FIGURES / 'd2_per_field_heatmap.png'}`
- `{OUTPUT_DIAGRAMS / 'd2_state_machine_v2.txt'}`

## Следующие шаги (Phase 2)
- Strategy A: Free-form LLM (CS-A × EM-3/EM-4/EM-6) — требует API key
- Strategy C: Hybrid SM+LLM (CS-C × EM-3/EM-4) — production-aligned
"""

report_path = OUTPUT_REPORTS / 'D2_phase1_summary.md'
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(report)

print(f"Отчёт сохранён: {report_path}")
print("\n" + "=" * 60)
print("PHASE 1 COMPLETE")
print("=" * 60)

Отчёт сохранён: /Users/kazdoraw/developer/med-agent/study/outputs/reports/D2_phase1_summary.md

PHASE 1 COMPLETE


---

## Phase 2: LLM Experiments (A3, C3)

**Стратегии:**
- **A3** (CS-A × EM-3): Free-form LLM — LLM сам решает что спрашивать, API JSON mode для extraction
- **C3** (CS-C × EM-3): Hybrid SM+LLM — SM определяет поле, LLM формулирует вопрос и извлекает значение

**Модель:** `meta-llama/Llama-3.3-70B-Instruct-Turbo` (production-aligned)
**Schema:** SD-1 (Universal 4 — production)
**Пациент:** тот же `simulate_patient_response()` что и в B1 (fair comparison)

> A4/C4 (Instructor) пропущены — `instructor` не установлен. Добавить в Phase 3 при необходимости.

In [57]:
# === LLM Client Setup (OpenRouter через openai SDK) ===
from openai import OpenAI
from pydantic import BaseModel, Field as PydField
import time as _time
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity as _cosine_sim

# --- Модель и параметры ---
LLM_MODEL = "qwen/qwen3-235b-a22b-2507"
LLM_TEMPERATURE = 0.1
LLM_MAX_TOKENS = 800
LLM_MAX_RETRIES = 1

_openrouter_key = os.getenv("OPENROUTER_API_KEY", "")
assert _openrouter_key, "OPENROUTER_API_KEY не установлен в .env"

llm_client = OpenAI(
    api_key=_openrouter_key,
    base_url="https://openrouter.ai/api/v1",
)

# --- Embedding model для semantic similarity ---
EMBEDDING_MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
_embedding_model = None

def _get_embedding_model() -> SentenceTransformer:
    """Lazy-load embedding model (загрузка один раз)."""
    global _embedding_model
    if _embedding_model is None:
        print(f"  Loading embedding model: {EMBEDDING_MODEL_NAME}...")
        _embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
        print(f"  Embedding model loaded ✓")
    return _embedding_model


# --- Pydantic schema: ВЫРОВНЕНА с production AnamnesisData ---
class IntakeExtraction(BaseModel):
    """Результат извлечения данных анамнеза (SD-1: 4 обязательных поля)."""
    symptoms: Optional[str] = PydField(default=None, description="Симптомы/жалоба пациента")
    localization: Optional[str] = PydField(default=None, description="Локализация (зуб, область)")
    duration: Optional[str] = PydField(default=None, description="Длительность проблемы")
    chronic_or_allergies: Optional[str] = PydField(default=None, description="Хронические заболевания/аллергии")
    missing_fields: List[str] = PydField(default_factory=list, description="Незаполненные поля")
    next_question: Optional[str] = PydField(default=None, description="Следующий вопрос пациенту")
    is_complete: bool = PydField(default=False, description="Все 4 поля заполнены")
    reasoning: Optional[str] = PydField(default=None, description="Краткое обоснование")


# --- LLM Stats Tracker ---
llm_stats: Dict[str, Any] = {
    "calls": 0, "tokens_in": 0, "tokens_out": 0,
    "errors": 0, "latency_ms": [],
}

def reset_llm_stats():
    """Сбросить статистику LLM вызовов."""
    llm_stats.update(calls=0, tokens_in=0, tokens_out=0, errors=0, latency_ms=[])


# --- Qwen3 <think> tag stripping ---
_THINK_TAG_RE = re.compile(r"<think>.*?</think>", re.DOTALL)

def _strip_think_tags(content: str) -> str:
    """Убрать <think>...</think> блоки из ответа Qwen3 thinking model."""
    return _THINK_TAG_RE.sub("", content).strip()


def _try_repair_json(raw: str) -> Optional[dict]:
    """Попытка починить truncated/malformed JSON."""
    raw = raw.strip()
    # Убрать markdown ```json ... ```
    if raw.startswith("```"):
        raw = re.sub(r"^```(?:json)?\s*", "", raw)
        raw = re.sub(r"\s*```$", "", raw)
    # Если обрезан — дополнить закрывающими скобками
    open_braces = raw.count("{") - raw.count("}")
    if open_braces > 0:
        if raw.rstrip()[-1] not in ('"', ',', '}', ']', 'e', 'l'):
            raw += '"'
        raw += "}" * open_braces
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return None


def call_llm_json(
    messages: List[Dict[str, str]],
    max_retries: int = LLM_MAX_RETRIES,
) -> Optional[IntakeExtraction]:
    """
    Вызов LLM с JSON mode + Pydantic validation.
    EM-3: API JSON mode → strip think tags → json.loads → Pydantic.
    """
    for attempt in range(max_retries + 1):
        try:
            t0 = _time.time()
            response = llm_client.chat.completions.create(
                model=LLM_MODEL,
                messages=messages,
                response_format={"type": "json_object"},
                temperature=LLM_TEMPERATURE,
                max_tokens=LLM_MAX_TOKENS,
            )
            latency_ms = (_time.time() - t0) * 1000

            llm_stats["calls"] += 1
            llm_stats["latency_ms"].append(latency_ms)
            if response.usage:
                llm_stats["tokens_in"] += response.usage.prompt_tokens
                llm_stats["tokens_out"] += response.usage.completion_tokens

            content = response.choices[0].message.content
            if content is None:
                llm_stats["errors"] += 1
                if attempt < max_retries:
                    _time.sleep(1)
                    continue
                return None

            # P4: Strip Qwen3 <think> tags before JSON parsing
            content = _strip_think_tags(content)

            try:
                data = json.loads(content)
            except json.JSONDecodeError:
                data = _try_repair_json(content)
                if data is None:
                    raise

            return IntakeExtraction(**data)

        except Exception as e:
            llm_stats["errors"] += 1
            if attempt == max_retries:
                print(f"  ⚠ LLM error (attempt {attempt+1}): {type(e).__name__}: {e}")
                return None
            _time.sleep(1)

    return None


# --- Маппинг вопрос → поле (для Strategy A) ---
QUESTION_FIELD_KEYWORDS: Dict[str, List[str]] = {
    "symptoms": ["беспокоит", "жалоб", "симптом", "боль", "проблем", "что случилось"],
    "localization": ["где", "место", "зуб", "область", "какой", "локализ"],
    "duration": ["давно", "когда", "сколько", "длительн", "продолж", "как долго"],
    "chronic_or_allergies": ["хронич", "аллерг", "заболеван", "лекарств", "болезн"],
}


def infer_target_field(question: str, missing_fields: List[str]) -> Optional[str]:
    """Определить целевое поле из вопроса LLM (keyword matching)."""
    q_lower = question.lower()
    for fld in missing_fields:
        keywords = QUESTION_FIELD_KEYWORDS.get(fld, [])
        if any(kw in q_lower for kw in keywords):
            return fld
    return missing_fields[0] if missing_fields else None


# --- Extraction Accuracy: embedding-based semantic similarity ---
SEMANTIC_THRESHOLD = 0.70  # Cosine similarity порог для "семантически эквивалентно"
EXACT_MATCH_BONUS = True   # Exact match всегда = 1.0

def compute_extraction_accuracy(
    filled: Dict[str, Any],
    gold: Dict[str, Any],
    required_fields: List[str],
    semantic_threshold: float = SEMANTIC_THRESHOLD,
) -> Dict[str, Any]:
    """
    Вычислить точность извлечения vs gold standard.
    
    Использует embedding similarity (sentence-transformers) вместо SequenceMatcher.
    Корректно обрабатывает семантическую эквивалентность:
    "справа внизу, дальний зуб" ≈ "нижний правый моляр" (cosine > 0.7)
    """
    model = _get_embedding_model()
    exact = 0
    semantic = 0
    per_field = {}
    
    # Собираем пары для batch encoding
    pairs_to_encode: List[Tuple[str, str, str]] = []  # (field, extracted, gold)
    
    for field in required_fields:
        extracted_val = str(filled.get(field, "")).strip()
        gold_val = str(gold.get(field, "")).strip()
        
        if not extracted_val or not gold_val:
            per_field[field] = {"match": "missing", "score": 0.0}
            continue
        
        # Exact match (case-insensitive)
        if extracted_val.lower() == gold_val.lower():
            exact += 1
            per_field[field] = {"match": "exact", "score": 1.0}
            continue
        
        pairs_to_encode.append((field, extracted_val, gold_val))
    
    # Batch embedding для оставшихся пар
    if pairs_to_encode:
        all_texts = []
        for _, ext, gld in pairs_to_encode:
            all_texts.extend([ext, gld])
        
        embeddings = model.encode(all_texts, normalize_embeddings=True)
        
        for i, (field, ext_val, gld_val) in enumerate(pairs_to_encode):
            emb_ext = embeddings[i * 2].reshape(1, -1)
            emb_gld = embeddings[i * 2 + 1].reshape(1, -1)
            sim = float(_cosine_sim(emb_ext, emb_gld)[0][0])
            
            if sim >= semantic_threshold:
                semantic += 1
                per_field[field] = {"match": "semantic", "score": round(sim, 3)}
            else:
                per_field[field] = {"match": "mismatch", "score": round(sim, 3)}
    
    total_with_gold = sum(1 for f in required_fields if gold.get(f))
    matched = exact + semantic
    
    return {
        "exact_matches": exact,
        "semantic_matches": semantic,
        "total_matched": matched,
        "total_with_gold": total_with_gold,
        "accuracy": matched / total_with_gold if total_with_gold > 0 else 0.0,
        "per_field": per_field,
    }


# Тест подключения
print(f"LLM Client: OpenRouter | Model: {LLM_MODEL}")
print(f"API key: {_openrouter_key[:15]}...")
print(f"Max tokens: {LLM_MAX_TOKENS} | Temperature: {LLM_TEMPERATURE}")
print(f"IntakeExtraction fields: {list(IntakeExtraction.model_fields.keys())}")
print(f"Extraction Accuracy: exact + embedding semantic (threshold={SEMANTIC_THRESHOLD})")
print(f"Qwen3 <think> tag stripping: enabled")
print(f"Embedding model: {EMBEDDING_MODEL_NAME} (lazy-loaded)")

LLM Client: OpenRouter | Model: qwen/qwen3-235b-a22b-2507
API key: sk-or-v1-293cb6...
Max tokens: 800 | Temperature: 0.1
IntakeExtraction fields: ['symptoms', 'localization', 'duration', 'chronic_or_allergies', 'missing_fields', 'next_question', 'is_complete', 'reasoning']
Extraction Accuracy: exact + embedding semantic (threshold=0.7)
Qwen3 <think> tag stripping: enabled
Embedding model: paraphrase-multilingual-MiniLM-L12-v2 (lazy-loaded)


### Strategy A: Free-form LLM (CS-A × EM-3)

LLM сам решает что спрашивать, без подсказки о missing fields.
На каждом ходу возвращает structured JSON с извлечёнными данными и следующим вопросом.

In [58]:
# === System prompt для Strategy A (Free-form LLM) ===
SYSTEM_PROMPT_A = """Ты — ИИ-ассистент стоматологической клиники. Собери анамнез пациента через диалог.

Обязательные поля:
1. symptoms — что беспокоит (симптомы/жалоба)
2. localization — где именно (зуб, область)
3. duration — как давно беспокоит
4. chronic_or_allergies — хронические заболевания, аллергии

Правила:
- Задавай по ОДНОМУ вопросу за раз
- Будь кратким и вежливым
- Извлекай данные из каждого ответа пациента
- Когда все 4 поля заполнены, установи is_complete=true
- Если пациент не знает — оставь поле null
- В missing_fields укажи список незаполненных полей
- НОРМАЛИЗУЙ данные: переводи разговорную речь в клинические термины
  Пример: "справа внизу, дальний зуб" → "нижний правый моляр"

Отвечай СТРОГО в JSON формате:
{
    "symptoms": "строка или null",
    "localization": "строка или null",
    "duration": "строка или null",
    "chronic_or_allergies": "строка или null",
    "missing_fields": ["незаполненные поля"],
    "next_question": "следующий вопрос пациенту (null если is_complete)",
    "is_complete": true/false,
    "reasoning": "какое поле заполняется этим вопросом"
}"""


def run_strategy_a(
    case: Dict[str, Any],
    schema_id: str = "SD-1",
    cooperation_rate: float = COOPERATION_RATE_DEFAULT,
    max_turns: int = MAX_TURNS_SM,
) -> Dict[str, Any]:
    """
    Strategy A: Free-form LLM (CS-A × EM-3).
    
    LLM сам решает порядок вопросов и извлекает данные.
    Assistant messages = natural language (вопрос), НЕ raw JSON.
    patient_answers → симулятор, gold_values → оценка.
    P3: поля обновляются если новое значение непустое (LLM может уточнить).
    """
    gold_values = map_gold_to_schema(
        case.get("gold_values", {}), case["target_complaint_type"], schema_id,
    )
    patient_answers = case.get("patient_answers", gold_values)
    
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_A},
        {"role": "user", "content": case["initial_user_message"]},
    ]
    dialog: List[Dict[str, str]] = [
        {"role": "patient", "text": case["initial_user_message"]}
    ]
    
    # P2: Единая инициализация — LLM извлекает ВСЕ поля, включая symptoms
    filled: Dict[str, Any] = {}
    turns = 0
    
    for _ in range(max_turns):
        extraction = call_llm_json(messages)
        if extraction is None:
            break
        
        # P3: Обновляем filled — новое непустое значение ПЕРЕЗАПИСЫВАЕТ старое
        for fld in SD1_FIELDS:
            val = getattr(extraction, fld, None)
            if val:
                filled[fld] = val
        
        if extraction.is_complete or not extraction.next_question:
            break
        
        question = extraction.next_question
        dialog.append({"role": "assistant", "text": question})
        messages.append({"role": "assistant", "content": question})
        
        missing = [f for f in SD1_FIELDS if f not in filled]
        if not missing:
            break
        
        target_field = infer_target_field(question, missing)
        response = simulate_patient_response(target_field, patient_answers, cooperation_rate)
        
        dialog.append({"role": "patient", "text": response})
        messages.append({"role": "user", "content": response})
        turns += 1
    
    required = SD1_FIELDS
    completion = sum(1 for f in required if f in filled) / len(required)
    accuracy_info = compute_extraction_accuracy(filled, gold_values, required)
    
    return {
        "case_id": case["case_id"],
        "schema_id": schema_id,
        "strategy": "A3",
        "complaint_type": case["target_complaint_type"],
        "target_complaint_type": case["target_complaint_type"],
        "filled_fields": filled,
        "required_fields": list(required),
        "turns": turns,
        "completion_rate": completion,
        "expert_sufficient": completion >= EXPERT_THRESHOLD,
        "extraction_accuracy": accuracy_info["accuracy"],
        "extraction_detail": accuracy_info,
        "stop_reason": "complete" if completion >= 1.0 else "max_turns",
        "dialog": dialog,
        "cooperation_rate": cooperation_rate,
    }


print("run_strategy_a() — P2: filled={}, P3: поля обновляются, нормализация в промпте.")

run_strategy_a() — P2: filled={}, P3: поля обновляются, нормализация в промпте.


In [59]:
# === Запуск Strategy A на всех кейсах (SD-1) ===
reset_llm_stats()
results_a: List[Dict] = []

print(f"Strategy A (CS-A × EM-3) | Model: {LLM_MODEL} | Cases: {N_CASES}")
print("-" * 60)

for i, case in enumerate(tqdm(cases, desc="A3")):
    result = run_strategy_a(case)
    results_a.append(result)
    
    # Прогресс каждые 10 кейсов
    if (i + 1) % 10 == 0:
        avg_c = np.mean([r["completion_rate"] for r in results_a])
        print(f"  [{i+1}/{N_CASES}] avg_completion={avg_c:.2f}, "
              f"llm_calls={llm_stats['calls']}, errors={llm_stats['errors']}")

# Сводка
completions_a = [r["completion_rate"] for r in results_a]
turns_a = [r["turns"] for r in results_a]
expert_a = [r["expert_sufficient"] for r in results_a]

print(f"\n=== Strategy A3 Results ===")
print(f"Completion: {np.mean(completions_a):.1%} ± {np.std(completions_a):.1%}")
print(f"Turns: {np.mean(turns_a):.1f} ± {np.std(turns_a):.1f}")
print(f"Expert Sufficient: {np.mean(expert_a):.1%}")
print(f"LLM calls: {llm_stats['calls']}, errors: {llm_stats['errors']}")
print(f"Tokens: {llm_stats['tokens_in']} in + {llm_stats['tokens_out']} out = {llm_stats['tokens_in'] + llm_stats['tokens_out']} total")
print(f"Avg latency: {np.mean(llm_stats['latency_ms']):.0f}ms")

# Сохраняем stats для A3
stats_a3 = llm_stats.copy()
stats_a3["latency_ms"] = list(stats_a3["latency_ms"])

Strategy A (CS-A × EM-3) | Model: qwen/qwen3-235b-a22b-2507 | Cases: 10
------------------------------------------------------------


A3:   0%|          | 0/10 [00:00<?, ?it/s]

  Loading embedding model: paraphrase-multilingual-MiniLM-L12-v2...


tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

  Embedding model loaded ✓


A3: 100%|██████████| 10/10 [09:52<00:00, 59.24s/it]

  [10/10] avg_completion=1.00, llm_calls=46, errors=2

=== Strategy A3 Results ===
Completion: 100.0% ± 0.0%
Turns: 3.4 ± 0.8
Expert Sufficient: 100.0%
LLM calls: 46, errors: 2
Tokens: 22222 in + 5784 out = 28006 total
Avg latency: 12582ms


### Strategy C: Hybrid SM+LLM (CS-C × EM-3)

SM определяет порядок полей (как B1), но LLM формулирует вопрос и извлекает значение из ответа.
Production-aligned: аналог `core/nodes/anamnesis.py` + `core/llm/provider.py:take_anamnesis()`.

In [60]:
# === System prompt для Strategy C (Hybrid SM+LLM) ===
# Production-aligned: один LLM вызов = извлечение + формулировка вопроса
SYSTEM_PROMPT_C = """Ты — ИИ-ассистент стоматологической клиники. Ведёшь сбор анамнеза.

Твоя задача:
1. Извлечь данные из ответа пациента в указанные поля
2. Сформулировать ОДИН вежливый вопрос для следующего незаполненного поля
3. НОРМАЛИЗУЙ данные: переводи разговорную речь в клинические термины
   Пример: "справа внизу, дальний зуб" → "нижний правый моляр"

Текущие данные анамнеза:
{current_data}

Целевое поле: {target_field} ({field_desc})
Незаполненные поля: {missing_list}

Последний вопрос ассистента: {last_question}
Ответ пациента: {user_text}

Отвечай СТРОГО в JSON формате:
{{
    "symptoms": "строка или null",
    "localization": "строка или null",
    "duration": "строка или null",
    "chronic_or_allergies": "строка или null",
    "missing_fields": ["незаполненные поля"],
    "next_question": "вопрос для следующего незаполненного поля",
    "is_complete": false,
    "reasoning": "что извлечено из ответа"
}}"""

FIELD_DESCRIPTIONS_C: Dict[str, str] = {
    "symptoms": "Основные симптомы и жалобы пациента",
    "localization": "Где именно болит или беспокоит (зуб, область, челюсть)",
    "duration": "Как давно беспокоит (дни, недели, месяцы)",
    "chronic_or_allergies": "Хронические заболевания и аллергии на лекарства",
}


def run_strategy_c(
    case: Dict[str, Any],
    schema_id: str = "SD-1",
    cooperation_rate: float = COOPERATION_RATE_DEFAULT,
    max_turns: int = MAX_TURNS_SM,
) -> Dict[str, Any]:
    """
    Strategy C: Hybrid SM+LLM (CS-C × EM-3).
    Production-aligned: 1 LLM вызов = извлечение данных + формулировка вопроса.
    
    P2: filled={} — LLM извлекает ВСЕ поля (включая symptoms из initial message).
    P3: поля обновляются при каждом LLM вызове (уточнение).
    patient_answers → симулятор, gold_values → оценка.
    """
    gold_values = map_gold_to_schema(
        case.get("gold_values", {}), case["target_complaint_type"], schema_id,
    )
    patient_answers = case.get("patient_answers", gold_values)
    required = SCHEMA_DESIGNS[schema_id]["fields_fn"](case["target_complaint_type"])
    
    dialog: List[Dict[str, str]] = [
        {"role": "patient", "text": case["initial_user_message"]}
    ]
    
    complaint_type = detect_complaint_type(case["initial_user_message"])
    # P2: Единая инициализация — LLM извлекает ВСЕ поля
    filled: Dict[str, Any] = {}
    
    turns = 0
    stop_reason = "max_turns"
    last_question = "Первый контакт"
    
    for _ in range(max_turns):
        missing = [f for f in required if f not in filled]
        if not missing:
            stop_reason = "complete"
            break
        
        target_field = missing[0]
        
        current_data_str = "\n".join(
            f"- {f}: {filled.get(f, 'не указано')}" for f in required
        )
        
        prompt = SYSTEM_PROMPT_C.format(
            current_data=current_data_str,
            target_field=target_field,
            field_desc=FIELD_DESCRIPTIONS_C.get(target_field, target_field),
            missing_list=", ".join(missing),
            last_question=last_question,
            user_text=dialog[-1]["text"] if dialog else "Начало диалога",
        )
        messages = [
            {"role": "system", "content": prompt},
            {"role": "user", "content": dialog[-1]["text"]},
        ]
        
        extraction = call_llm_json(messages)
        if extraction is None:
            question = SD1_QUESTIONS.get(target_field, f"Расскажите о {target_field}")
        else:
            # P3: Обновляем filled — новое непустое значение ПЕРЕЗАПИСЫВАЕТ старое
            for fld in required:
                val = getattr(extraction, fld, None)
                if val:
                    filled[fld] = val
            question = extraction.next_question or SD1_QUESTIONS.get(
                target_field, f"Расскажите о {target_field}"
            )
        
        missing_after = [f for f in required if f not in filled]
        if not missing_after:
            stop_reason = "complete"
            break
        
        dialog.append({"role": "assistant", "text": question})
        last_question = question
        
        next_target = missing_after[0]
        response = simulate_patient_response(next_target, patient_answers, cooperation_rate)
        dialog.append({"role": "patient", "text": response})
        turns += 1
        
        # Regex fallback: если LLM не извлёк целевое поле
        if next_target not in filled:
            val = extract_field_value_regex(response, next_target)
            if val:
                filled[next_target] = val
    
    completion = sum(1 for f in required if f in filled) / len(required) if required else 1.0
    accuracy_info = compute_extraction_accuracy(filled, gold_values, required)
    
    return {
        "case_id": case["case_id"],
        "schema_id": schema_id,
        "strategy": "C3",
        "complaint_type": complaint_type,
        "target_complaint_type": case["target_complaint_type"],
        "filled_fields": filled,
        "required_fields": list(required),
        "turns": turns,
        "completion_rate": completion,
        "expert_sufficient": completion >= EXPERT_THRESHOLD,
        "extraction_accuracy": accuracy_info["accuracy"],
        "extraction_detail": accuracy_info,
        "stop_reason": stop_reason,
        "dialog": dialog,
        "cooperation_rate": cooperation_rate,
    }


print("run_strategy_c() — P2: filled={}, P3: поля обновляются, нормализация в промпте.")

run_strategy_c() — P2: filled={}, P3: поля обновляются, нормализация в промпте.


In [61]:
# === Запуск Strategy C на всех кейсах (SD-1) ===
reset_llm_stats()
results_c: List[Dict] = []

print(f"Strategy C (CS-C × EM-3) | Model: {LLM_MODEL} | Cases: {N_CASES}")
print("-" * 60)

for i, case in enumerate(tqdm(cases, desc="C3")):
    result = run_strategy_c(case)
    results_c.append(result)
    
    if (i + 1) % 10 == 0:
        avg_c = np.mean([r["completion_rate"] for r in results_c])
        print(f"  [{i+1}/{N_CASES}] avg_completion={avg_c:.2f}, "
              f"llm_calls={llm_stats['calls']}, errors={llm_stats['errors']}")

# Сводка
completions_c = [r["completion_rate"] for r in results_c]
turns_c = [r["turns"] for r in results_c]
expert_c = [r["expert_sufficient"] for r in results_c]

print(f"\n=== Strategy C3 Results ===")
print(f"Completion: {np.mean(completions_c):.1%} ± {np.std(completions_c):.1%}")
print(f"Turns: {np.mean(turns_c):.1f} ± {np.std(turns_c):.1f}")
print(f"Expert Sufficient: {np.mean(expert_c):.1%}")
print(f"LLM calls: {llm_stats['calls']}, errors: {llm_stats['errors']}")
print(f"Tokens: {llm_stats['tokens_in']} in + {llm_stats['tokens_out']} out = {llm_stats['tokens_in'] + llm_stats['tokens_out']} total")
print(f"Avg latency: {np.mean(llm_stats['latency_ms']):.0f}ms")

stats_c3 = llm_stats.copy()
stats_c3["latency_ms"] = list(stats_c3["latency_ms"])

Strategy C (CS-C × EM-3) | Model: qwen/qwen3-235b-a22b-2507 | Cases: 10
------------------------------------------------------------


C3: 100%|██████████| 10/10 [04:23<00:00, 26.32s/it]

  [10/10] avg_completion=1.00, llm_calls=29, errors=0

=== Strategy C3 Results ===
Completion: 100.0% ± 0.0%
Turns: 2.9 ± 0.3
Expert Sufficient: 100.0%
LLM calls: 29, errors: 0
Tokens: 13085 in + 5920 out = 19005 total
Avg latency: 9015ms


### Phase 2: Сравнение стратегий (A3 vs B1 vs C3)

In [62]:
# === Сравнительная таблица: A3 vs B1 (SD-1) vs C3 ===
# B1 берём из Phase 1 results (SD-1)
completions_b1 = [r["completion_rate"] for r in results_b["SD-1"]]
turns_b1 = [r["turns"] for r in results_b["SD-1"]]
expert_b1 = [r["expert_sufficient"] for r in results_b["SD-1"]]

# Extraction Accuracy для B1 (вычисляем здесь, т.к. функция определена в Phase 2)
accuracy_b1 = []
for i, r in enumerate(results_b["SD-1"]):
    gold_mapped = map_gold_to_schema(
        cases[i].get("gold_values", {}),
        cases[i]["target_complaint_type"],
        "SD-1",
    )
    acc_info = compute_extraction_accuracy(
        r["filled_fields"], gold_mapped, r["required_fields"]
    )
    accuracy_b1.append(acc_info["accuracy"])

# Extraction Accuracy для A3 и C3 (уже вычислена в run_strategy_a/c)
accuracy_a3 = [r.get("extraction_accuracy", 0.0) for r in results_a]
accuracy_c3 = [r.get("extraction_accuracy", 0.0) for r in results_c]

strategies = {
    "A3 (Free-form LLM)": {
        "completions": completions_a, "turns": turns_a, "expert": expert_a,
        "accuracy": accuracy_a3,
    },
    "B1 (Deterministic SM)": {
        "completions": completions_b1, "turns": turns_b1, "expert": expert_b1,
        "accuracy": accuracy_b1,
    },
    "C3 (Hybrid SM+LLM)": {
        "completions": completions_c, "turns": turns_c, "expert": expert_c,
        "accuracy": accuracy_c3,
    },
}

print(f"=== Сравнение стратегий (SD-1, cooperation={COOPERATION_RATE_DEFAULT}) ===")
print(f"Extraction metric: embedding similarity (threshold={SEMANTIC_THRESHOLD})\n")
print(f"{'Strategy':<25} {'Completion':>12} {'Accuracy':>10} {'Turns':>8} {'Expert %':>10} {'Token Cost':>12}")
print("-" * 80)

for name, data in strategies.items():
    token_cost = "0"
    if "A3" in name:
        token_cost = f"{stats_a3['tokens_in'] + stats_a3['tokens_out']:,}"
    elif "C3" in name:
        token_cost = f"{stats_c3['tokens_in'] + stats_c3['tokens_out']:,}"
    
    print(f"{name:<25} {np.mean(data['completions']):>11.1%} {np.mean(data['accuracy']):>9.1%} "
          f"{np.mean(data['turns']):>8.1f} {np.mean(data['expert']):>9.1%} {token_cost:>12}")

# DataFrame для дальнейшего анализа — включает cooperation_mode
comparison_rows = []
for name, data in strategies.items():
    for i in range(len(data["completions"])):
        comparison_rows.append({
            "strategy": name.split(" (")[0],
            "case_id": cases[i]["case_id"],
            "complaint_type": cases[i]["target_complaint_type"],
            "cooperation_mode": cases[i].get("cooperation_mode", "cooperative"),
            "completion_rate": data["completions"][i],
            "extraction_accuracy": data["accuracy"][i],
            "turns": data["turns"][i],
            "expert_sufficient": data["expert"][i],
        })

df_comparison = pd.DataFrame(comparison_rows)
print(f"\nDataFrame shape: {df_comparison.shape}")
print(f"\nExtraction Accuracy breakdown:")
for name in ["A3", "B1", "C3"]:
    subset = df_comparison[df_comparison["strategy"] == name]
    coop = subset[subset["cooperation_mode"] == "cooperative"]
    adv = subset[subset["cooperation_mode"] != "cooperative"]
    print(f"  {name}: overall={subset['extraction_accuracy'].mean():.1%}, "
          f"cooperative={coop['extraction_accuracy'].mean():.1%}" +
          (f", adversarial={adv['extraction_accuracy'].mean():.1%}" if len(adv) > 0 else ""))

=== Сравнение стратегий (SD-1, cooperation=0.9) ===
Extraction metric: embedding similarity (threshold=0.7)

Strategy                    Completion   Accuracy    Turns   Expert %   Token Cost
--------------------------------------------------------------------------------
A3 (Free-form LLM)             100.0%     62.5%      3.4    100.0%       28,006
B1 (Deterministic SM)           95.0%     30.8%      2.9     90.0%            0
C3 (Hybrid SM+LLM)             100.0%     50.0%      2.9    100.0%       19,005

DataFrame shape: (30, 8)

Extraction Accuracy breakdown:
  A3: overall=62.5%, cooperative=62.5%
  B1: overall=30.8%, cooperative=30.8%
  C3: overall=50.0%, cooperative=50.0%


In [63]:
# === Визуализация: Strategy Comparison (4 панели) ===
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

strategy_names = ["A3", "B1", "C3"]
strategy_colors = {"A3": "#e74c3c", "B1": "#1f77b4", "C3": "#2ca02c"}

# --- 1. Completion Rate boxplot ---
ax1 = axes[0, 0]
box_data_comp = [df_comparison[df_comparison["strategy"] == s]["completion_rate"].values for s in strategy_names]
bp1 = ax1.boxplot(box_data_comp, tick_labels=strategy_names, patch_artist=True)
for patch, s in zip(bp1["boxes"], strategy_names):
    patch.set_facecolor(strategy_colors[s])
    patch.set_alpha(0.6)
ax1.axhline(y=0.9, color="green", linestyle="--", alpha=0.7, label="Target 90%")
ax1.set_ylabel("Completion Rate")
ax1.set_title("Completion Rate по стратегиям")
ax1.set_ylim(0, 1.1)
ax1.legend()

# --- 2. Extraction Accuracy boxplot ---
ax2 = axes[0, 1]
box_data_acc = [df_comparison[df_comparison["strategy"] == s]["extraction_accuracy"].values for s in strategy_names]
bp2 = ax2.boxplot(box_data_acc, tick_labels=strategy_names, patch_artist=True)
for patch, s in zip(bp2["boxes"], strategy_names):
    patch.set_facecolor(strategy_colors[s])
    patch.set_alpha(0.6)
ax2.axhline(y=0.7, color="orange", linestyle="--", alpha=0.7, label="Target 70%")
ax2.set_ylabel("Extraction Accuracy")
ax2.set_title("Extraction Accuracy vs Gold (embedding similarity)")
ax2.set_ylim(0, 1.1)
ax2.legend()

# --- 3. Turns boxplot ---
ax3 = axes[1, 0]
box_data_turns = [df_comparison[df_comparison["strategy"] == s]["turns"].values for s in strategy_names]
bp3 = ax3.boxplot(box_data_turns, tick_labels=strategy_names, patch_artist=True)
for patch, s in zip(bp3["boxes"], strategy_names):
    patch.set_facecolor(strategy_colors[s])
    patch.set_alpha(0.6)
ax3.set_ylabel("Turns")
ax3.set_title("Ходы по стратегиям")

# --- 4. Expert Sufficiency bar ---
ax4 = axes[1, 1]
expert_rates = [np.mean(strategies[k]["expert"]) for k in strategies]
bars = ax4.bar(strategy_names, expert_rates, color=[strategy_colors[s] for s in strategy_names], alpha=0.7)
ax4.axhline(y=0.7, color="green", linestyle="--", alpha=0.7, label="Target 70%")
ax4.set_ylabel("Expert Sufficiency Rate")
ax4.set_title("Expert Sufficiency")
ax4.set_ylim(0, 1.1)
ax4.legend()
for bar, val in zip(bars, expert_rates):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f"{val:.0%}",
             ha="center", fontsize=11, fontweight="bold")

plt.tight_layout()
fig_path = OUTPUT_FIGURES / "d2_strategy_comparison_phase2.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Сохранено: {fig_path}")

Сохранено: /Users/kazdoraw/developer/med-agent/study/outputs/figures/d2_strategy_comparison_phase2.png


/var/folders/6w/j5zxb2n50dj_hrb7t96x08bh0000gn/T/ipykernel_87878/3220800276.py:59: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
# === Export Phase 2: Metrics CSV + Dialogs JSONL + Report ===

# --- P5: Экспорт диалогов в JSONL для анализа вопросов LLM ---
dialogs_path = OUTPUT_TABLES / "d2_phase2_dialogs.jsonl"
with open(dialogs_path, "w", encoding="utf-8") as f:
    for name, strat_key, results_list in [
        ("A3", "A3", results_a),
        ("B1", "B1", results_b["SD-1"]),
        ("C3", "C3", results_c),
    ]:
        for r in results_list:
            record = {
                "strategy": strat_key,
                "case_id": r["case_id"],
                "complaint_type": r.get("target_complaint_type", r.get("complaint_type")),
                "completion_rate": r["completion_rate"],
                "extraction_accuracy": r.get("extraction_accuracy", 0.0),
                "extraction_detail": r.get("extraction_detail", {}),
                "filled_fields": r["filled_fields"],
                "turns": r["turns"],
                "stop_reason": r.get("stop_reason", "unknown"),
                "dialog": r.get("dialog", []),
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
print(f"Диалоги: {dialogs_path} ({sum(len(r) for r in [results_a, results_b['SD-1'], results_c])} записей)")

# --- Metrics CSV ---
phase2_rows = []
for name, strat_key, results_list, stats in [
    ("A3 (Free-form LLM)", "A3", results_a, stats_a3),
    ("B1 (Deterministic SM)", "B1", results_b["SD-1"], {"tokens_in": 0, "tokens_out": 0, "calls": 0, "errors": 0}),
    ("C3 (Hybrid SM+LLM)", "C3", results_c, stats_c3),
]:
    completions = [r["completion_rate"] for r in results_list]
    turns = [r["turns"] for r in results_list]
    expert = [r["expert_sufficient"] for r in results_list]
    accuracy = [r.get("extraction_accuracy", 0.0) for r in results_list]
    
    if strat_key == "B1":
        accuracy = accuracy_b1
    
    phase2_rows.append({
        "strategy": strat_key,
        "strategy_name": name,
        "schema": "SD-1",
        "n_cases": len(results_list),
        "avg_completion": round(np.mean(completions), 4),
        "std_completion": round(np.std(completions), 4),
        "avg_accuracy": round(np.mean(accuracy), 4),
        "std_accuracy": round(np.std(accuracy), 4),
        "avg_turns": round(np.mean(turns), 2),
        "std_turns": round(np.std(turns), 2),
        "expert_sufficient_rate": round(np.mean(expert), 4),
        "total_tokens": stats["tokens_in"] + stats["tokens_out"],
        "llm_calls": stats["calls"],
        "llm_errors": stats["errors"],
    })

df_phase2 = pd.DataFrame(phase2_rows)
phase2_path = OUTPUT_TABLES / "d2_phase2_strategy_comparison.csv"
df_phase2.to_csv(phase2_path, index=False)
print(f"Phase 2 metrics: {phase2_path}")
print(df_phase2.to_string(index=False))

# Per-case comparison
comparison_path = OUTPUT_TABLES / "d2_phase2_per_case.csv"
df_comparison.to_csv(comparison_path, index=False)
print(f"\nPer-case comparison: {comparison_path}")

# --- Phase 2 report ---
phase2_report = f"""# D2 Phase 2: LLM Experiments — Результаты

**Дата:** {datetime.now().strftime('%Y-%m-%d %H:%M')}
**Модель:** {LLM_MODEL}
**Schema:** SD-1 (Universal 4)
**Cases:** {N_CASES} | **Cooperation rate:** {COOPERATION_RATE_DEFAULT}
**Extraction Accuracy:** embedding semantic similarity (threshold={SEMANTIC_THRESHOLD})
**Embedding model:** {EMBEDDING_MODEL_NAME}

## Рефакторинг v2

### Исправленные проблемы
1. **P1**: SequenceMatcher → embedding similarity (sentence-transformers)
2. **P2**: Единая инициализация filled={{}} (LLM извлекает ВСЕ поля)
3. **P3**: Поля обновляются при уточнении (не "first-write wins")
4. **P4**: Qwen3 `<think>` tag stripping
5. **P5**: Экспорт диалогов в JSONL для анализа
6. **Промпт**: Инструкция нормализации (разговорная → клиническая)
7. **Adversarial**: 5 adversarial шаблонов в CASE_TEMPLATES

## Сравнение стратегий

| Strategy | Completion | Accuracy | Turns | Expert Suff. | Tokens | Errors |
|----------|-----------|----------|-------|-------------|--------|--------|
"""
for row in phase2_rows:
    phase2_report += (
        f"| {row['strategy_name']} "
        f"| {row['avg_completion']:.1%} ± {row['std_completion']:.1%} "
        f"| {row['avg_accuracy']:.1%} ± {row['std_accuracy']:.1%} "
        f"| {row['avg_turns']:.1f} ± {row['std_turns']:.1f} "
        f"| {row['expert_sufficient_rate']:.1%} "
        f"| {row['total_tokens']:,} "
        f"| {row['llm_errors']} |\n"
    )

phase2_report += f"""
## Артефакты
- Metrics: `{phase2_path.name}`
- Per-case: `{comparison_path.name}`
- Dialogs: `{dialogs_path.name}`
- Figure: `d2_strategy_comparison_phase2.png`
"""

report_path = OUTPUT_REPORTS / "D2_phase2_summary.md"
with open(report_path, "w", encoding="utf-8") as f:
    f.write(phase2_report)
print(f"\nОтчёт: {report_path}")
print("\n" + "=" * 60)
print("PHASE 2 COMPLETE")
print("=" * 60)

Диалоги: /Users/kazdoraw/developer/med-agent/study/outputs/tables/d2_phase2_dialogs.jsonl (30 записей)
Phase 2 metrics: /Users/kazdoraw/developer/med-agent/study/outputs/tables/d2_phase2_strategy_comparison.csv
strategy         strategy_name schema  n_cases  avg_completion  std_completion  avg_accuracy  std_accuracy  avg_turns  std_turns  expert_sufficient_rate  total_tokens  llm_calls  llm_errors
      A3    A3 (Free-form LLM)   SD-1       10            1.00            0.00        0.6250        0.2275        3.4        0.8                     1.0         28006         46           2
      B1 B1 (Deterministic SM)   SD-1       10            0.95            0.15        0.3083        0.2142        2.9        0.3                     0.9             0          0           0
      C3    C3 (Hybrid SM+LLM)   SD-1       10            1.00            0.00        0.5000        0.3249        2.9        0.3                     1.0         19005         29           0

Per-case comparison: /Users/

: 